# Healthcare Challenge 2 - Baseline Submission

This notebook provides a simple baseline for **Healthcare Challenge 2: ED Cost Prediction**.

**Goal**: Predict `ed_cost_next3y_usd` for each patient
**Metric**: Mean Absolute Error (MAE) - Lower is better

## Instructions:
1. **Replace API credentials** in the first cell with your team's API key and name
2. **Run all cells** to generate and submit baseline predictions
3. **Check the output** for your submission score

This baseline uses only tabular ED cost data with a simple Random Forest regressor.


In [ ]:
# 1. Initialize Client and Load Data

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from agentds import BenchmarkClient

# # 🔑 REPLACE WITH YOUR CREDENTIALS
# client = BenchmarkClient(
#     api_key="your_api_key",        # Get from your team dashboard
#     team_name="your_team_name"     # Your exact team name
# )

# BEST FINAL CODE
- MAE: 447.9542

In [9]:
# ============================================
# CODE 43 | Based on Code31 (Standalone) - EXACT Code31 skeleton
# SINGLE CHANGE: Add Model C (depth=3, stronger regularization) to A/B ensemble
#   - depth=3, l2_leaf_reg=8, min_data_in_leaf=40, random_strength=3.0
#   - Uses PRUNED features (like B but shallower/more regularized)
#   - 3-way weight search (A,B,C) with same one-SE + bag logic
# Rationale: More regularized shallow model adds diversity without new features.
#   Should reduce variance in ensemble, potentially shrink CV-LB gap.
# Everything else is IDENTICAL to Code31 (features, receipts, chronic shift, etc.)
# ============================================

import os, re, sys, gc, math, time, random, zlib, warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None
pd.set_option("display.max_columns", 260)
pd.set_option("display.width", 220)

# -----------------------------
# Config (IDENTICAL to Code31 except 3-way ensemble)
# -----------------------------
class CFG:
    FAST_MODE = True
    N_FOLDS = 7
    N_SEEDS = 3 if FAST_MODE else 5
    TRIM_K = 0 if N_SEEDS < 5 else 1

    ITERS = 3200
    ES_ROUNDS = 120
    LR = 0.03
    RSM = 0.80
    SUBSAMPLE = 0.80

    USE_FULLFIT_BLEND = True
    FULLFIT_BLEND_W = 0.35

    # 3-way weight search (CHANGED from 2-way)
    W_STEP_3WAY = 0.05
    STD_PEN = 0.20
    ONE_SE_TOL = 0.08
    # For bagging: we bag near the chosen point in A-B-C simplex

    SHIFT_FOLDS = 5
    CHRONIC_FACTORS = [0.0, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75]
    K_CHRONIC = 250.0
    CAP_CHRONIC = 250.0
    PEN_CHRONIC_ON = 0.02
    MIN_IMPROVE_TO_APPLY = 0.05

    USE_BASELINE_SHRINK = True
    SHRINK_GRID_STEP = 0.05
    SHRINK_ONESE_TOL = 0.20
    MIN_GAIN_TO_APPLY = 0.05

    TASK_TYPE = "CPU"
    GPU_BORDER_COUNT = 128

    REQUIRE_RECEIPTS = True
    MIN_RCPT_FEATURES = 20
    RCPT_MATCH_TOL = 1e-3
    RCPT_MATCH_RATE_MIN = 0.99

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

ID_COL = "patient_id"
TARGET = "ed_cost_next3y_usd"

DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"
TRAIN_PATH = os.path.join(DATA_DIR, "ed_cost_train.csv")
TEST_PATH  = os.path.join(DATA_DIR, "ed_cost_test.csv")
PATIENTS_PATH = os.path.join(DATA_DIR, "patients.csv")
ADM_TRAIN_PATH = os.path.join(DATA_DIR, "admissions_train.csv")
ADM_TEST_PATH  = os.path.join(DATA_DIR, "admissions_test.csv")
RECEIPTS_JOBLIB_PATH = os.path.join(DATA_DIR, "receipts_parsed.joblib")
OUT_SUB_PATH = os.path.join(DATA_DIR, "submission.csv")

# -----------------------------
# Minimal deps
# -----------------------------
def _pip_install(pkg: str):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    from joblib import load as joblib_load
except Exception:
    _pip_install("joblib")
    from joblib import load as joblib_load

try:
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error
except Exception:
    _pip_install("scikit-learn")
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error

try:
    from catboost import CatBoostRegressor
except Exception:
    _pip_install("catboost")
    from catboost import CatBoostRegressor

# -----------------------------
# Utilities (IDENTICAL to Code31)
# -----------------------------
def must_exist(path: str, name: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {name}: {path}")

def log_shape(name: str, df: pd.DataFrame):
    mem = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"[{name}] shape={df.shape} | cols={df.shape[1]} | mem={mem:.2f} MB")

def qdict(x, qs=(0,0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99,1.0)):
    x = np.asarray(x, dtype=float)
    return {q: float(np.quantile(x, q)) for q in qs}

def standardize_zip3(z):
    if z is None or (isinstance(z, float) and np.isnan(z)):
        return None
    s = re.sub(r"\D", "", str(z).strip())
    if not s:
        return None
    return s.zfill(3)

def norm_code(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip().upper()
    if s == "" or s.lower() == "nan":
        return None
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    s = re.sub(r"\s+", "", s)
    return s

def safe_num_series(s: pd.Series, default=0.0) -> pd.Series:
    v = pd.to_numeric(s, errors="coerce")
    v = v.replace([np.inf, -np.inf], np.nan).fillna(default)
    return v

def stable_hash(s: str) -> int:
    return int(zlib.crc32(s.encode("utf-8")) & 0xffffffff)

def gini(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if x.size == 0:
        return 0.0
    if np.all(x == 0):
        return 0.0
    x = np.sort(x)
    n = x.size
    cumx = np.cumsum(x)
    return float((n + 1 - 2*np.sum(cumx)/cumx[-1]) / n)

def trimmed_mean(mat: np.ndarray, trim_k: int = 1) -> np.ndarray:
    mat = np.asarray(mat, dtype=float)
    if mat.ndim != 2:
        raise ValueError("trimmed_mean expects 2D array")
    n = mat.shape[0]
    if trim_k <= 0 or n < 2*trim_k + 1:
        return np.mean(mat, axis=0)
    s = np.sort(mat, axis=0)
    return np.mean(s[trim_k:n-trim_k, :], axis=0)

# -----------------------------
# Admissions aggregates (IDENTICAL to Code31)
# -----------------------------
def load_admissions_features(adm_train_path: str, adm_test_path: str) -> Optional[pd.DataFrame]:
    dfs = []
    for path in [adm_train_path, adm_test_path]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            if "readmit_30d" in df.columns:
                df = df.drop(columns=["readmit_30d"], errors="ignore")
            dfs.append(df)
    if not dfs:
        return None

    adm = pd.concat(dfs, ignore_index=True)

    need = ["patient_id","charlson_band","acuity_emergent"]
    if not all(c in adm.columns for c in need):
        print("[warn] admissions missing required cols -> skip admissions features")
        return None

    adm["patient_id"] = pd.to_numeric(adm["patient_id"], errors="coerce").astype("Int64")
    adm = adm.dropna(subset=["patient_id"]).copy()
    adm["patient_id"] = adm["patient_id"].astype(int)

    adm["charlson_band"] = pd.to_numeric(adm["charlson_band"], errors="coerce")
    adm["acuity_emergent"] = pd.to_numeric(adm["acuity_emergent"], errors="coerce")

    out = adm.groupby("patient_id").agg(
        charlson_max=("charlson_band","max"),
        charlson_mean=("charlson_band","mean"),
        pct_emergent=("acuity_emergent","mean"),
    ).reset_index()

    for c in ["charlson_max","charlson_mean","pct_emergent"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0.0)

    return out

# -----------------------------
# Receipts features (IDENTICAL to Code31)
# -----------------------------
def build_receipt_features_from_lineitems(li: pd.DataFrame) -> pd.DataFrame:
    li = li.copy()

    code_col = None
    for c in ["code","cpt","cpt_code","hcpcs","proc_code","procedure_code"]:
        if c in li.columns:
            code_col = c
            break

    total_col = None
    for c in ["line_total","line_total_usd","total","amount","line_cost","sum_items","item_total","extended_price","sum_line_total"]:
        if c in li.columns:
            total_col = c
            break

    unit_col = None
    for c in ["unit_price","unitprice","unit_cost","unitcost"]:
        if c in li.columns:
            unit_col = c
            break

    if "patient_id" not in li.columns or code_col is None or total_col is None:
        raise ValueError("receipts lineitems missing required columns.")

    li["patient_id"] = pd.to_numeric(li["patient_id"], errors="coerce").astype("Int64")
    li = li.dropna(subset=["patient_id"]).copy()
    li["patient_id"] = li["patient_id"].astype(int)

    li["code"] = li[code_col].map(norm_code)
    li = li.dropna(subset=["code"]).copy()

    li["amt"] = pd.to_numeric(li[total_col], errors="coerce").fillna(0.0).astype(float)
    li.loc[li["amt"] < 0, "amt"] = 0.0

    if unit_col is not None:
        li["unit_price"] = pd.to_numeric(li[unit_col], errors="coerce").replace([np.inf,-np.inf], np.nan)
    else:
        li["unit_price"] = np.nan

    rcpt_sum = li.groupby("patient_id")["amt"].sum().rename("rcpt_sum_items")
    li = li.join(rcpt_sum, on="patient_id")
    denom = li["rcpt_sum_items"].replace(0.0, np.nan)
    share = (li["amt"] / denom).replace([np.inf,-np.inf], np.nan).fillna(0.0)

    cost_hhi = (share * share).groupby(li["patient_id"]).sum().rename("cost_hhi")
    top1_share = share.groupby(li["patient_id"]).max().rename("top1_share")

    def _topk_sum(s, k=3):
        a = np.sort(s.values.astype(float))[::-1]
        return float(a[:k].sum()) if a.size else 0.0
    top3_share = share.groupby(li["patient_id"]).apply(lambda s: _topk_sum(s, 3)).rename("top3_share")

    gini_amt = li.groupby("patient_id")["amt"].apply(lambda s: gini(s.values)).rename("gini_line_total")
    max_line_total = li.groupby("patient_id")["amt"].max().rename("max_line_total")

    code_num = pd.to_numeric(li["code"].where(li["code"].str.fullmatch(r"\d+"), None), errors="coerce")

    em_codes = ["99281","99282","99283","99284","99285"]
    is_em = li["code"].isin(em_codes)
    em_map = {"99281":1,"99282":2,"99283":3,"99284":4,"99285":5}
    em_level = li["code"].map(em_map).fillna(0).astype(float)

    is_crit = li["code"].isin(["99291","99292"])
    is_obs = li["code"].str.startswith("G037", na=False)
    is_high = li["code"].isin(["31500","36556","32551","36620","92950"])

    is_lab = code_num.between(80000, 89999)
    is_imaging = code_num.between(70000, 79999)
    is_proc_general = code_num.between(10000, 69999)
    is_proc_any = is_high | (is_proc_general & (~is_high) & (~is_em) & (~is_crit))

    n_unique_codes = li.groupby("patient_id")["code"].nunique().rename("n_unique_codes")
    n_em_codes = is_em.astype(int).groupby(li["patient_id"]).sum().rename("n_em_codes")
    max_em_level = em_level.groupby(li["patient_id"]).max().rename("max_em_level")
    sum_em_level = (em_level * is_em.astype(int)).groupby(li["patient_id"]).sum().rename("sum_em_level")
    avg_em_level = (sum_em_level / n_em_codes.replace(0, np.nan)).fillna(0.0).rename("avg_em_level")
    n_high_em = ((em_level >= 4) & is_em).astype(int).groupby(li["patient_id"]).sum().rename("n_high_em")

    em_total = li.loc[is_em].groupby("patient_id")["amt"].sum().rename("em_total")
    crit_total = li.loc[is_crit].groupby("patient_id")["amt"].sum().rename("crit_total")
    proc_total = li.loc[is_proc_any].groupby("patient_id")["amt"].sum().rename("proc_total")
    img_total = li.loc[is_imaging].groupby("patient_id")["amt"].sum().rename("img_total")
    lab_total = li.loc[is_lab].groupby("patient_id")["amt"].sum().rename("lab_total")
    high_total = li.loc[is_high].groupby("patient_id")["amt"].sum().rename("high_total")

    n_procedures = is_proc_any.astype(int).groupby(li["patient_id"]).sum().rename("n_procedures")
    n_imaging = is_imaging.astype(int).groupby(li["patient_id"]).sum().rename("n_imaging")
    n_lab = is_lab.astype(int).groupby(li["patient_id"]).sum().rename("n_lab")

    has_critical_care = is_crit.astype(int).groupby(li["patient_id"]).max().rename("has_critical_care")
    has_high_acuity = is_high.astype(int).groupby(li["patient_id"]).max().rename("has_high_acuity")
    has_observation = is_obs.astype(int).groupby(li["patient_id"]).max().rename("has_observation")
    has_imaging = is_imaging.astype(int).groupby(li["patient_id"]).max().rename("has_imaging")

    def has_code(code: str, name: str):
        return (li["code"].eq(code).astype(int).groupby(li["patient_id"]).max()).rename(name)

    has_intub_31500 = has_code("31500","has_intub_31500")
    has_cvc_36556 = has_code("36556","has_cvc_36556")
    has_cpr_92950 = has_code("92950","has_cpr_92950")
    has_artline_36620 = has_code("36620","has_artline_36620")
    has_ct_head_70450 = has_code("70450","has_ct_head_70450")
    has_99285 = has_code("99285","has_99285")
    has_ct_abdpel_74177 = has_code("74177","has_ct_abdpel_74177")
    has_troponin_84484 = has_code("84484","has_troponin_84484")
    has_obs_G0378 = has_code("G0378","has_obs_G0378")

    med_unit_all = li.loc[li["unit_price"].notna()].groupby("patient_id")["unit_price"].median().rename("median_unit_price")

    def _median_unit(mask, name):
        sub = li.loc[mask & li["unit_price"].notna(), ["patient_id","unit_price"]]
        if sub.empty:
            return None
        return sub.groupby("patient_id")["unit_price"].median().rename(name)

    med_unit_em  = _median_unit(is_em, "median_unit_price_em")
    med_unit_img = _median_unit(is_imaging, "median_unit_price_imaging")
    med_unit_lab = _median_unit(is_lab, "median_unit_price_lab")

    out = pd.concat([
        rcpt_sum,
        n_unique_codes, cost_hhi, top1_share, top3_share, gini_amt, max_line_total,
        n_em_codes, max_em_level, avg_em_level, n_high_em,
        has_critical_care, has_high_acuity, has_observation, has_imaging,
        has_intub_31500, has_cvc_36556, has_cpr_92950, has_artline_36620,
        has_ct_head_70450, has_99285, has_ct_abdpel_74177, has_troponin_84484, has_obs_G0378,
        n_procedures, n_imaging, n_lab
    ], axis=1).reset_index()

    for s in [em_total, crit_total, proc_total, img_total, lab_total, high_total]:
        out = out.merge(s.reset_index(), on="patient_id", how="left")

    for c in ["em_total","crit_total","proc_total","img_total","lab_total","high_total","rcpt_sum_items"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    denom2 = out["rcpt_sum_items"].replace(0.0, np.nan)
    out["pct_cost_em"] = (out["em_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_procedure"] = (out["proc_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_critical"] = (out["crit_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_imaging"] = (out["img_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_lab"] = (out["lab_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_high_acuity"] = (out["high_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)

    out["cost_per_em"] = np.where(out["n_em_codes"] > 0,
                                  out["rcpt_sum_items"] / out["n_em_codes"].clip(lower=1),
                                  out["rcpt_sum_items"])
    out["n_high_acuity_total"] = (
        out["has_intub_31500"].fillna(0).astype(int)
        + out["has_cvc_36556"].fillna(0).astype(int)
        + out["has_cpr_92950"].fillna(0).astype(int)
        + out["has_artline_36620"].fillna(0).astype(int)
        + out["has_critical_care"].fillna(0).astype(int)
    ).astype(int)

    out = out.merge(med_unit_all.reset_index(), on="patient_id", how="left")
    if med_unit_em is not None:  out = out.merge(med_unit_em.reset_index(), on="patient_id", how="left")
    else: out["median_unit_price_em"] = np.nan
    if med_unit_img is not None: out = out.merge(med_unit_img.reset_index(), on="patient_id", how="left")
    else: out["median_unit_price_imaging"] = np.nan
    if med_unit_lab is not None: out = out.merge(med_unit_lab.reset_index(), on="patient_id", how="left")
    else: out["median_unit_price_lab"] = np.nan

    for c in ["median_unit_price","median_unit_price_em","median_unit_price_imaging","median_unit_price_lab","max_line_total"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0.0)
        out["log1p_" + c] = np.log1p(out[c].clip(lower=0.0))

    out.drop(columns=[c for c in ["em_total","crit_total","proc_total","img_total","lab_total","high_total"] if c in out.columns],
             inplace=True, errors="ignore")

    for c in out.columns:
        if c == "patient_id":
            continue
        out[c] = pd.to_numeric(out[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0.0)

    return out

def load_receipts_joblib(joblib_path: str) -> Optional[pd.DataFrame]:
    if not os.path.exists(joblib_path):
        return None
    data = joblib_load(joblib_path)

    if isinstance(data, pd.DataFrame):
        df = data
        if ("patient_id" in df.columns) and any(c in df.columns for c in ["code","cpt","cpt_code","hcpcs","proc_code","procedure_code"]):
            try:
                return build_receipt_features_from_lineitems(df)
            except Exception:
                pass
        if "patient_id" in df.columns:
            return df
        return None

    if isinstance(data, dict):
        cand_keys = ["lineitems_df","lineitems","items_df","items","line_items_df","line_items","df","data"]
        for k in cand_keys:
            if k in data and isinstance(data[k], pd.DataFrame):
                df = data[k]
                if ("patient_id" in df.columns) and any(c in df.columns for c in ["code","cpt","cpt_code","hcpcs","proc_code","procedure_code"]):
                    return build_receipt_features_from_lineitems(df)
                if "patient_id" in df.columns:
                    return df
        for v in data.values():
            if isinstance(v, pd.DataFrame) and "patient_id" in v.columns:
                df = v
                if any(c in df.columns for c in ["code","cpt","cpt_code","hcpcs","proc_code","procedure_code"]):
                    return build_receipt_features_from_lineitems(df)
                return df
        return None

    if isinstance(data, (list, tuple)):
        dfs = [x for x in data if isinstance(x, pd.DataFrame)]
        for df in dfs:
            if ("patient_id" in df.columns) and any(c in df.columns for c in ["code","cpt","cpt_code","hcpcs","proc_code","procedure_code"]):
                return build_receipt_features_from_lineitems(df)
        for df in dfs:
            if "patient_id" in df.columns:
                return df
        return None

    return None

# -----------------------------
# Feature engineering (IDENTICAL to Code31)
# -----------------------------
def build_features(ed_df: pd.DataFrame,
                   patients_df: pd.DataFrame,
                   adm_df: Optional[pd.DataFrame],
                   rcpt_df: Optional[pd.DataFrame]) -> pd.DataFrame:
    feat = ed_df.copy()

    chronic_map = {"PNEUMONIA":0, "DIABETESCOMP":1, "HF":2}
    feat["primary_chronic"] = feat["primary_chronic"].astype(str)
    feat["chronic_encoded"] = feat["primary_chronic"].str.upper().map(chronic_map).fillna(-1).astype(float)

    feat["prior_ed_visits_5y"] = safe_num_series(feat["prior_ed_visits_5y"], default=0.0).clip(lower=0.0)
    feat["prior_ed_cost_5y_usd"] = safe_num_series(feat["prior_ed_cost_5y_usd"], default=0.0).clip(lower=0.0)

    feat["prior_cost_cap20k"] = feat["prior_ed_cost_5y_usd"].clip(upper=20000.0)
    feat["sqrt_prior_cost"] = np.sqrt(feat["prior_ed_cost_5y_usd"].clip(lower=0.0))
    feat["log_prior_cost"] = np.log1p(feat["prior_ed_cost_5y_usd"].clip(lower=0.0))
    feat["log_prior_cost_cap20k"] = np.log1p(feat["prior_cost_cap20k"].clip(lower=0.0))
    feat["log_visits"] = np.log1p(feat["prior_ed_visits_5y"].clip(lower=0.0))
    feat["cost_per_visit"] = feat["prior_ed_cost_5y_usd"] / feat["prior_ed_visits_5y"].clip(lower=1.0)

    feat["baseline_next3y"] = feat["prior_ed_cost_5y_usd"] * (3.0/5.0)

    p = patients_df.copy()
    p["patient_id"] = pd.to_numeric(p["patient_id"], errors="coerce").astype("Int64")
    p = p.dropna(subset=["patient_id"]).copy()
    p["patient_id"] = p["patient_id"].astype(int)

    p["age"] = pd.to_numeric(p["age"], errors="coerce").fillna(p["age"].median()).clip(lower=0, upper=110)
    p["sex_encoded"] = (p["sex"].astype(str).str.upper() == "M").astype(int)

    ins = p["insurance"].astype(str).str.lower()
    ins_map = {"private":2, "public":1, "self_pay":0, "selfpay":0}
    p["insurance_encoded"] = ins.map(ins_map).fillna(-1).astype(float)

    z3 = p["zip3"].apply(standardize_zip3).astype("string")
    zr = z3.fillna("000").str.replace(r"\D","", regex=True).str.zfill(3).str[0]
    p["zip_region"] = pd.to_numeric(zr, errors="coerce").fillna(-1).astype(float)

    feat = feat.merge(p[["patient_id","age","sex_encoded","insurance_encoded","zip_region"]], on="patient_id", how="left")
    feat["ins_x_chronic"] = feat["insurance_encoded"].fillna(-1) * feat["chronic_encoded"].fillna(-1)

    if adm_df is not None:
        feat = feat.merge(adm_df, on="patient_id", how="left")
        for c in ["charlson_max","charlson_mean","pct_emergent"]:
            if c in feat.columns:
                feat[c] = pd.to_numeric(feat[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0.0)

    if rcpt_df is not None:
        feat = feat.merge(rcpt_df, on="patient_id", how="left")
        for c in rcpt_df.columns:
            if c == "patient_id":
                continue
            feat[c] = pd.to_numeric(feat[c], errors="coerce").replace([np.inf,-np.inf], np.nan)
            if feat[c].isna().any():
                feat[c] = feat[c].fillna(feat[c].median() if not feat[c].isna().all() else 0.0)

    if "pct_cost_critical" in feat.columns:
        feat["logprior_x_pctcritical"] = feat["log_prior_cost"] * feat["pct_cost_critical"]
    if "n_high_acuity_total" in feat.columns:
        feat["logprior_x_highacu"] = feat["log_prior_cost"] * feat["n_high_acuity_total"]

    if "n_unique_codes" in feat.columns:
        feat["cost_per_code"] = feat["prior_ed_cost_5y_usd"] / feat["n_unique_codes"].clip(lower=1.0)

    feat.replace([np.inf, -np.inf], np.nan, inplace=True)
    for c in feat.columns:
        if c in ["patient_id", "primary_chronic", TARGET]:
            continue
        feat[c] = pd.to_numeric(feat[c], errors="coerce")
        if feat[c].isna().any():
            feat[c] = feat[c].fillna(feat[c].median() if not feat[c].isna().all() else 0.0)

    return feat

def get_numeric_feature_cols(df: pd.DataFrame) -> List[str]:
    exclude = {"patient_id","primary_chronic",TARGET,"sex","insurance","zip3"}
    return [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]

def drop_constants(cols: List[str], df: pd.DataFrame) -> List[str]:
    return [c for c in cols if c in df.columns and df[c].nunique(dropna=False) > 1]

# -----------------------------
# Training (A/B/C) - Code31 style + Model C
# -----------------------------
def _adjust_params_for_task(params: Dict) -> Dict:
    p = dict(params)
    if CFG.TASK_TYPE.upper() == "GPU":
        p.pop("rsm", None)
        p["border_count"] = CFG.GPU_BORDER_COUNT
    return p

def train_models(train_feat: pd.DataFrame, test_feat: pd.DataFrame,
                 feat_full: List[str], feat_pruned: List[str],
                 strat_vec: np.ndarray):
    y = train_feat[TARGET].values.astype(float)

    # =============================================
    # SINGLE CHANGE: Added Model C (depth=3, stronger regularization)
    # A and B are IDENTICAL to Code31
    # =============================================
    model_specs = {
        "A_full_d5": dict(
            cols=feat_full,
            params=dict(
                loss_function="RMSE", eval_metric="MAE",
                depth=5, l2_leaf_reg=5, min_data_in_leaf=28,
                learning_rate=CFG.LR, iterations=CFG.ITERS,
                rsm=CFG.RSM, bootstrap_type="Bernoulli", subsample=CFG.SUBSAMPLE,
                random_strength=1.0,
            )
        ),
        "B_pruned_d4": dict(
            cols=feat_pruned,
            params=dict(
                loss_function="RMSE", eval_metric="MAE",
                depth=4, l2_leaf_reg=4, min_data_in_leaf=32,
                learning_rate=CFG.LR, iterations=CFG.ITERS,
                rsm=CFG.RSM, bootstrap_type="Bernoulli", subsample=CFG.SUBSAMPLE,
                random_strength=2.0,
            )
        ),
        # NEW: Model C - shallower, more regularized, for diversity
        "C_pruned_d3": dict(
            cols=feat_pruned,
            params=dict(
                loss_function="RMSE", eval_metric="MAE",
                depth=3, l2_leaf_reg=8, min_data_in_leaf=40,
                learning_rate=CFG.LR, iterations=CFG.ITERS,
                rsm=0.70, bootstrap_type="Bernoulli", subsample=0.75,
                random_strength=3.0,
            )
        ),
    }

    oof_by_seed = {m: [] for m in model_specs}
    test_by_seed = {m: [] for m in model_specs}
    best_iters = {m: [] for m in model_specs}

    print(f"\n[training] CatBoost {CFG.TASK_TYPE} | folds={CFG.N_FOLDS} seeds={CFG.N_SEEDS} | iters={CFG.ITERS} es={CFG.ES_ROUNDS}")
    print("Models:", list(model_specs.keys()), "| TRIM_K:", CFG.TRIM_K)

    t_all = time.time()
    for sidx in range(CFG.N_SEEDS):
        rs = SEED + sidx * 7
        kf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=rs)

        oof_seed = {m: np.zeros(len(train_feat), dtype=float) for m in model_specs}
        test_seed_foldbag = {m: np.zeros(len(test_feat), dtype=float) for m in model_specs}
        best_iters_seed = {m: [] for m in model_specs}

        for fold, (ti, vi) in enumerate(kf.split(train_feat, strat_vec), 1):
            for mname, spec in model_specs.items():
                cols = spec["cols"]
                params = _adjust_params_for_task(spec["params"])

                X_tr = train_feat[cols].iloc[ti]
                y_tr = y[ti]
                X_va = train_feat[cols].iloc[vi]
                y_va = y[vi]
                X_te = test_feat[cols]

                cb = CatBoostRegressor(
                    **params,
                    task_type=CFG.TASK_TYPE,
                    thread_count=-1,
                    verbose=0,
                    allow_writing_files=False,
                    random_seed=int(rs + fold * 31 + stable_hash(mname) % 997),
                    early_stopping_rounds=CFG.ES_ROUNDS,
                )
                cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=0)

                try:
                    bi = int(cb.get_best_iteration())
                except Exception:
                    bi = None
                if bi is not None and bi > 0:
                    best_iters[mname].append(bi)
                    best_iters_seed[mname].append(bi)

                pred_va = np.clip(cb.predict(X_va).astype(float), 0, None)
                pred_te = np.clip(cb.predict(X_te).astype(float), 0, None)

                oof_seed[mname][vi] = pred_va
                test_seed_foldbag[mname] += pred_te / CFG.N_FOLDS

                del cb
                gc.collect()

        test_seed_final = {}
        if CFG.USE_FULLFIT_BLEND:
            for mname, spec in model_specs.items():
                cols = spec["cols"]
                params = _adjust_params_for_task(spec["params"])

                X_all = train_feat[cols]
                X_te = test_feat[cols]

                it_med = int(np.median(best_iters_seed[mname])) if best_iters_seed[mname] else int(CFG.ITERS * 0.45)
                it_use = int(max(350, min(CFG.ITERS, it_med + 150)))
                params["iterations"] = it_use

                cb_full = CatBoostRegressor(
                    **params,
                    task_type=CFG.TASK_TYPE,
                    thread_count=-1,
                    verbose=0,
                    allow_writing_files=False,
                    random_seed=int(rs + 999 + stable_hash("FULL_"+mname) % 997),
                )
                cb_full.fit(X_all, y, verbose=0)
                pred_te_full = np.clip(cb_full.predict(X_te).astype(float), 0, None)

                test_seed_final[mname] = (1.0 - CFG.FULLFIT_BLEND_W) * test_seed_foldbag[mname] + CFG.FULLFIT_BLEND_W * pred_te_full

                del cb_full
                gc.collect()
        else:
            test_seed_final = test_seed_foldbag

        seed_maes = {m: float(mean_absolute_error(y, oof_seed[m])) for m in model_specs}
        print(f"  Seed {sidx+1}/{CFG.N_SEEDS} OOF MAE: " + " | ".join([f"{m}={seed_maes[m]:.2f}" for m in model_specs]))

        for m in model_specs:
            oof_by_seed[m].append(oof_seed[m])
            test_by_seed[m].append(test_seed_final[m])

    print("\n[seed-aggregated OOF MAE per model] (trimmed mean)")
    for m in model_specs:
        avg_oof = trimmed_mean(np.vstack(oof_by_seed[m]), trim_k=CFG.TRIM_K)
        print(f"  {m:15s}: {mean_absolute_error(y, avg_oof):.2f}")

    print("\n[median best_iteration] (ref)")
    for m in model_specs:
        print(f"  {m:15s}: {int(np.median(best_iters[m])) if best_iters[m] else '(n/a)'}")

    print(f"\n[training] total time: {time.time()-t_all:.1f}s")
    return oof_by_seed, test_by_seed, best_iters

# =============================================
# SINGLE CHANGE: 3-way weight search on A,B,C simplex
# =============================================
def weight_search_3way(oof_by_seed: Dict[str, List[np.ndarray]], y: np.ndarray):
    """Search over (wA, wB, wC) simplex where wA+wB+wC=1, step=0.05"""
    step = CFG.W_STEP_3WAY
    grid_1d = np.round(np.arange(0.0, 1.0 + 1e-9, step), 10)

    rows = []
    for wA in grid_1d:
        for wB in grid_1d:
            wC = round(1.0 - wA - wB, 10)
            if wC < -1e-9 or wC > 1.0 + 1e-9:
                continue
            wC = max(0.0, wC)
            maes = []
            for s in range(CFG.N_SEEDS):
                p = wA*oof_by_seed["A_full_d5"][s] + wB*oof_by_seed["B_pruned_d4"][s] + wC*oof_by_seed["C_pruned_d3"][s]
                maes.append(float(mean_absolute_error(y, p)))
            mean_m = float(np.mean(maes))
            std_m = float(np.std(maes))
            obj = mean_m + CFG.STD_PEN*std_m
            rows.append((obj, mean_m, std_m, float(wA), float(wB), float(wC)))

    rows.sort(key=lambda x: x[0])

    best_obj = rows[0][0]
    
    # Also find best AB-only (wC=0) for comparison
    ab_only = [r for r in rows if abs(r[5]) < 1e-9]
    ab_only.sort(key=lambda x: x[0])
    best_ab_obj = ab_only[0][0] if ab_only else None

    print(f"\n[3-way ensemble search] Total combos: {len(rows)} | Top 15 (obj=mean+{CFG.STD_PEN}*std):")
    for i, r in enumerate(rows[:15], 1):
        obj, mean_m, std_m, wA, wB, wC = r
        marker = " <-- AB-only" if abs(wC) < 1e-9 else ""
        print(f"  #{i:02d} obj={obj:.3f} mean={mean_m:.3f} std={std_m:.3f} | wA={wA:.2f} wB={wB:.2f} wC={wC:.2f}{marker}")

    if best_ab_obj is not None:
        print(f"\n[AB-only best] obj={ab_only[0][0]:.3f} mean={ab_only[0][1]:.3f} | wA={ab_only[0][3]:.2f} wB={ab_only[0][4]:.2f}")
        print(f"[ABC   best]   obj={rows[0][0]:.3f} mean={rows[0][1]:.3f} | wA={rows[0][3]:.2f} wB={rows[0][4]:.2f} wC={rows[0][5]:.2f}")
        abc_gain = ab_only[0][0] - rows[0][0]
        print(f"[ABC gain over AB-only] {abc_gain:+.3f} obj")

    # One-SE selection: prefer simpler (smaller wC, then smaller wA)
    eligible = [r for r in rows if r[0] <= best_obj + CFG.ONE_SE_TOL]
    # Sort by: wC ascending (less model C = simpler), then wA ascending, then obj
    chosen = min(eligible, key=lambda r: (r[5], r[3], r[0]))
    ch_obj, ch_mean, ch_std, ch_wA, ch_wB, ch_wC = chosen

    print(f"\n[one-SE selection] chosen: obj={ch_obj:.3f} | wA={ch_wA:.2f} wB={ch_wB:.2f} wC={ch_wC:.2f} | tol={CFG.ONE_SE_TOL:.2f}")

    # If chosen wC=0, model C didn't help -> fall back to AB-only
    if abs(ch_wC) < 1e-9:
        print("[NOTE] one-SE chose wC=0 -> Model C not contributing. Falling back to AB-only ensemble.")

    meta = {
        "best": {"wA":rows[0][3], "wB":rows[0][4], "wC":rows[0][5], "obj":rows[0][0], "mean":rows[0][1], "std":rows[0][2]},
        "chosen": {"wA":ch_wA, "wB":ch_wB, "wC":ch_wC, "obj":ch_obj, "mean":ch_mean, "std":ch_std},
        "best_ab_only": {"wA":ab_only[0][3], "wB":ab_only[0][4], "obj":ab_only[0][0], "mean":ab_only[0][1]} if ab_only else None,
    }
    return meta

def build_3way_bag_preds(oof_by_seed, test_by_seed, chosen_meta: Dict, y: np.ndarray) -> Tuple[np.ndarray, np.ndarray, Dict]:
    """Build bagged predictions around the chosen 3-way weight point.
    We bag over small perturbations of the chosen weights on the simplex."""
    yA = np.vstack(oof_by_seed["A_full_d5"])
    yB = np.vstack(oof_by_seed["B_pruned_d4"])
    yC = np.vstack(oof_by_seed["C_pruned_d3"])
    tA = np.vstack(test_by_seed["A_full_d5"])
    tB = np.vstack(test_by_seed["B_pruned_d4"])
    tC = np.vstack(test_by_seed["C_pruned_d3"])

    ch = chosen_meta["chosen"]
    wA0, wB0, wC0 = ch["wA"], ch["wB"], ch["wC"]
    step = CFG.W_STEP_3WAY

    # Generate bag points: chosen + neighbors on simplex within 1 step
    bag_points = set()
    for dA in [-step, 0, step]:
        for dB in [-step, 0, step]:
            wA = round(wA0 + dA, 10)
            wB = round(wB0 + dB, 10)
            wC = round(1.0 - wA - wB, 10)
            if wA >= -1e-9 and wB >= -1e-9 and wC >= -1e-9 and wA <= 1+1e-9 and wB <= 1+1e-9 and wC <= 1+1e-9:
                bag_points.add((max(0,wA), max(0,wB), max(0,wC)))

    bag_points = sorted(bag_points)
    print(f"\n[3-way weight-bagging] {len(bag_points)} points around chosen ({wA0:.2f},{wB0:.2f},{wC0:.2f})")

    oof_preds = []
    test_preds = []
    per_point_mae = {}

    for (wA, wB, wC) in bag_points:
        oof_mat = wA*yA + wB*yB + wC*yC
        test_mat = wA*tA + wB*tB + wC*tC

        oof_trim = trimmed_mean(oof_mat, trim_k=CFG.TRIM_K)
        test_trim = trimmed_mean(test_mat, trim_k=CFG.TRIM_K)

        oof_preds.append(oof_trim)
        test_preds.append(test_trim)
        key = f"({wA:.2f},{wB:.2f},{wC:.2f})"
        per_point_mae[key] = float(mean_absolute_error(y, oof_trim))

    oof_bag = np.mean(np.vstack(oof_preds), axis=0)
    test_bag = np.mean(np.vstack(test_preds), axis=0)

    # Also compute AB-only bag for comparison (Code31 equivalent)
    ab_meta = chosen_meta.get("best_ab_only")
    ab_oof_mae = None
    if ab_meta:
        wA_ab, wB_ab = ab_meta["wA"], ab_meta["wB"]
        ab_oof = trimmed_mean(wA_ab*yA + wB_ab*yB, trim_k=CFG.TRIM_K)
        ab_oof_mae = float(mean_absolute_error(y, ab_oof))

    meta = {"per_point_oof_mae": per_point_mae, "n_bag_points": len(bag_points), "ab_only_oof_mae": ab_oof_mae}
    return oof_bag, test_bag, meta

# -----------------------------
# Chronic residual shift (IDENTICAL to Code31)
# -----------------------------
def _shrink(n: int, k: float) -> float:
    return float(n / (n + k)) if n > 0 else 0.0

def fit_chronic_shifts(y_tr: np.ndarray, p_tr: np.ndarray, chronic_tr: np.ndarray, cf: float) -> Dict[str, float]:
    y_tr = np.asarray(y_tr, dtype=float)
    p_tr = np.asarray(p_tr, dtype=float)
    chronic_tr = np.asarray(chronic_tr).astype(str)
    resid = y_tr - p_tr

    shifts = {}
    for g in np.unique(chronic_tr):
        m = (chronic_tr == g)
        n = int(m.sum())
        med = float(np.median(resid[m])) if n > 0 else 0.0
        val = cf * _shrink(n, CFG.K_CHRONIC) * med
        shifts[g] = float(np.clip(val, -CFG.CAP_CHRONIC, CFG.CAP_CHRONIC))
    return shifts

def apply_chronic_shifts(p: np.ndarray, chronic: np.ndarray, shifts: Dict[str, float]) -> np.ndarray:
    p = np.asarray(p, dtype=float).copy()
    chronic = np.asarray(chronic).astype(str)
    for g, sh in shifts.items():
        p[chronic == g] += sh
    return p

def tune_chronic_shift_cv(y: np.ndarray, p_base: np.ndarray, chronic: np.ndarray) -> Dict:
    y = np.asarray(y, dtype=float)
    p_base = np.asarray(p_base, dtype=float)
    chronic = np.asarray(chronic).astype(str)

    base_mae = float(mean_absolute_error(y, p_base))
    strat = LabelEncoder().fit_transform(chronic)
    kf = StratifiedKFold(n_splits=CFG.SHIFT_FOLDS, shuffle=True, random_state=SEED)

    rows = []
    best = None

    for cf in CFG.CHRONIC_FACTORS:
        pcv = np.zeros_like(p_base, dtype=float)
        for tr_idx, va_idx in kf.split(p_base, strat):
            shifts = fit_chronic_shifts(y[tr_idx], p_base[tr_idx], chronic[tr_idx], cf)
            pcv[va_idx] = apply_chronic_shifts(p_base[va_idx], chronic[va_idx], shifts)
        mae = float(mean_absolute_error(y, pcv))
        obj = mae + (CFG.PEN_CHRONIC_ON if cf > 0 else 0.0)
        rows.append((obj, mae, cf))
        if best is None or obj < best[0]:
            best = (obj, mae, cf)

    rows.sort(key=lambda x: x[0])
    print("\n[chronic shift tuning] Top 10 (obj=CV_MAE+penalty):")
    for i, (obj, mae, cf) in enumerate(rows[:10], 1):
        print(f"  #{i:02d} obj={obj:.3f} | CV_MAE={mae:.3f} | chronic_factor={cf:.2f}")

    obj, mae, cf = best
    meta = {"base_mae": base_mae, "obj": float(obj), "cv_mae": float(mae), "cf": float(cf), "cv_gain_vs_base": float(base_mae - mae)}
    return meta

# -----------------------------
# Baseline shrink (IDENTICAL to Code31)
# -----------------------------
def tune_baseline_shrink(oof_pred: np.ndarray, y: np.ndarray, baseline: np.ndarray) -> Dict:
    grid = np.round(np.arange(0.0, 1.0 + 1e-9, CFG.SHRINK_GRID_STEP), 10)
    rows = []
    for w in grid:
        p = w*oof_pred + (1.0-w)*baseline
        mae = float(mean_absolute_error(y, p))
        rows.append((mae, float(w)))
    rows.sort(key=lambda x: x[0])
    best_mae, best_w = rows[0]

    eligible = [r for r in rows if r[0] <= best_mae + CFG.SHRINK_ONESE_TOL]
    chosen_mae, chosen_w = min(eligible, key=lambda r: (r[1], r[0]))

    meta = {
        "best_mae": float(best_mae),
        "best_w_model": float(best_w),
        "chosen_mae": float(chosen_mae),
        "chosen_w_model": float(chosen_w),
        "grid_step": float(CFG.SHRINK_GRID_STEP),
        "tol": float(CFG.SHRINK_ONESE_TOL),
        "top5": [(float(m), float(w)) for (m,w) in rows[:5]]
    }
    return meta

def apply_baseline_shrink(pred: np.ndarray, baseline: np.ndarray, w_model: float) -> np.ndarray:
    return w_model*np.asarray(pred, float) + (1.0-w_model)*np.asarray(baseline, float)

# -----------------------------
# RUN
# -----------------------------
print("="*110)
print("CODE 43 | Code31 + SINGLE CHANGE: Add Model C (depth=3, l2=8, min_leaf=40, rs=3.0) to ensemble")
print("="*110)
print(f"[cfg] FAST_MODE={CFG.FAST_MODE} | folds={CFG.N_FOLDS} seeds={CFG.N_SEEDS} trim_k={CFG.TRIM_K} | task={CFG.TASK_TYPE}")
print(f"[paths] DATA_DIR={DATA_DIR}")
print()

must_exist(TRAIN_PATH, "ed_cost_train.csv")
must_exist(TEST_PATH, "ed_cost_test.csv")
must_exist(PATIENTS_PATH, "patients.csv")
must_exist(ADM_TRAIN_PATH, "admissions_train.csv")
must_exist(ADM_TEST_PATH, "admissions_test.csv")
if CFG.REQUIRE_RECEIPTS:
    must_exist(RECEIPTS_JOBLIB_PATH, "receipts_parsed.joblib")

t0 = time.time()
print("[load] reading CSVs...")
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
patients = pd.read_csv(PATIENTS_PATH)

log_shape("ed_cost_train", train)
log_shape("ed_cost_test", test)
log_shape("patients", patients)

print("\n[target stats]")
print(train[TARGET].describe().to_string())

for df in [train, test, patients]:
    df[ID_COL] = pd.to_numeric(df[ID_COL], errors="coerce").astype(int)

print("\n[admissions] building aggregates...")
adm_df = load_admissions_features(ADM_TRAIN_PATH, ADM_TEST_PATH)
print("  admissions features:", None if adm_df is None else (adm_df.shape, list(adm_df.columns)))

print("\n[receipts] building receipt features...")
rcpt_df = None
if os.path.exists(RECEIPTS_JOBLIB_PATH):
    try:
        rcpt_df = load_receipts_joblib(RECEIPTS_JOBLIB_PATH)
        if rcpt_df is not None:
            rcpt_df["patient_id"] = pd.to_numeric(rcpt_df["patient_id"], errors="coerce").astype("Int64")
            rcpt_df = rcpt_df.dropna(subset=["patient_id"]).copy()
            rcpt_df["patient_id"] = rcpt_df["patient_id"].astype(int)
            rcpt_df = rcpt_df.drop_duplicates("patient_id", keep="last")

            n_feats = rcpt_df.shape[1] - 1
            print(f"  rcpt_df shape: {rcpt_df.shape} | n_features={n_feats}")

            if CFG.REQUIRE_RECEIPTS and n_feats < CFG.MIN_RCPT_FEATURES:
                raise RuntimeError(f"Receipts features too few ({n_feats}) < MIN_RCPT_FEATURES={CFG.MIN_RCPT_FEATURES}. STOP.")
        else:
            if CFG.REQUIRE_RECEIPTS:
                raise RuntimeError("Receipts joblib loaded but rcpt_df is None. STOP.")
            else:
                print("  [warn] receipts missing -> continue without receipts.")
    except Exception as e:
        if CFG.REQUIRE_RECEIPTS:
            raise
        print(f"  [warn] receipts build failed: {e}")
        rcpt_df = None
else:
    if CFG.REQUIRE_RECEIPTS:
        raise FileNotFoundError(f"Missing receipts: {RECEIPTS_JOBLIB_PATH}")

print("\n[features] building train/test feature frames...")
train_feat = build_features(train, patients, adm_df, rcpt_df)
test_feat  = build_features(test,  patients, adm_df, rcpt_df)

# receipts invariant check
if rcpt_df is not None and ("rcpt_sum_items" in train_feat.columns):
    diff = (train_feat["rcpt_sum_items"].values.astype(float) - train_feat["prior_ed_cost_5y_usd"].values.astype(float))
    ok = np.isfinite(diff) & (np.abs(diff) <= CFG.RCPT_MATCH_TOL)
    match = float(np.mean(ok))
    print(f"  Receipt rcpt_sum_items vs prior_ed_cost_5y_usd match_rate(train): {match:.4f}")
    if CFG.REQUIRE_RECEIPTS and match < CFG.RCPT_MATCH_RATE_MIN:
        raise RuntimeError(f"Receipt invariant mismatch (match_rate={match:.4f} < {CFG.RCPT_MATCH_RATE_MIN}). STOP.")
if rcpt_df is not None and ("rcpt_sum_items" in test_feat.columns):
    diff = (test_feat["rcpt_sum_items"].values.astype(float) - test_feat["prior_ed_cost_5y_usd"].values.astype(float))
    ok = np.isfinite(diff) & (np.abs(diff) <= CFG.RCPT_MATCH_TOL)
    match = float(np.mean(ok))
    print(f"  Receipt rcpt_sum_items vs prior_ed_cost_5y_usd match_rate(test):  {match:.4f}")
    if CFG.REQUIRE_RECEIPTS and match < CFG.RCPT_MATCH_RATE_MIN:
        raise RuntimeError(f"Receipt invariant mismatch in test (match_rate={match:.4f} < {CFG.RCPT_MATCH_RATE_MIN}). STOP.")

feat_full = drop_constants(get_numeric_feature_cols(train_feat), train_feat)
feat_full = [c for c in feat_full if c != TARGET]
if "rcpt_sum_items" in feat_full:
    feat_full = [c for c in feat_full if c != "rcpt_sum_items"]

pruned_candidates = [
    "prior_ed_visits_5y","prior_ed_cost_5y_usd","prior_cost_cap20k","sqrt_prior_cost","log_prior_cost","log_prior_cost_cap20k",
    "cost_per_visit","log_visits","baseline_next3y",
    "chronic_encoded","age","sex_encoded","insurance_encoded","zip_region","ins_x_chronic",
    "charlson_max","charlson_mean","pct_emergent",
    "cost_per_em","cost_hhi","pct_cost_em","pct_cost_procedure","pct_cost_critical","pct_cost_high_acuity","pct_cost_imaging","pct_cost_lab",
    "n_high_acuity_total","has_critical_care","has_99285","max_em_level","avg_em_level","n_high_em","n_unique_codes",
    "top1_share","top3_share","gini_line_total","max_line_total",
    "median_unit_price","median_unit_price_em","median_unit_price_imaging","median_unit_price_lab",
    "log1p_median_unit_price","log1p_median_unit_price_em","log1p_median_unit_price_imaging","log1p_median_unit_price_lab","log1p_max_line_total",
    "logprior_x_pctcritical","logprior_x_highacu","cost_per_code",
]
feat_pruned = drop_constants([c for c in pruned_candidates if c in train_feat.columns], train_feat)
if "rcpt_sum_items" in feat_pruned:
    feat_pruned = [c for c in feat_pruned if c != "rcpt_sum_items"]

print(f"\n[feature counts] FULL={len(feat_full)} | PRUNED={len(feat_pruned)}")
print(f"[feature names FULL]  (sorted): {sorted(feat_full)}")
print(f"[feature names PRUNED] (sorted): {sorted(feat_pruned)}")

for c in set(feat_full + feat_pruned + ["baseline_next3y"]):
    if c not in train_feat.columns:
        continue
    med = train_feat[c].median() if not train_feat[c].isna().all() else 0.0
    train_feat[c] = pd.to_numeric(train_feat[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(med)
    if c in test_feat.columns:
        test_feat[c]  = pd.to_numeric(test_feat[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(med)

y = train_feat[TARGET].values.astype(float)

tmp = train_feat[["primary_chronic", TARGET]].copy()
tmp["bin"] = pd.qcut(tmp[TARGET], q=5, labels=False, duplicates="drop").astype(str)
tmp["strat"] = tmp["primary_chronic"].astype(str) + "_" + tmp["bin"]
strat_vec = LabelEncoder().fit_transform(tmp["strat"].values)

# train A/B/C
oof_by_seed, test_by_seed, best_iters = train_models(train_feat, test_feat, feat_full, feat_pruned, strat_vec)

# 3-way weight search
wmeta = weight_search_3way(oof_by_seed, y)

# 3-way bag
oof_base, test_base, bag_meta = build_3way_bag_preds(oof_by_seed, test_by_seed, wmeta, y=y)

base_mae = float(mean_absolute_error(y, oof_base))
print("\n[base ensemble after 3-way weight-bagging]")
print("  bagged OOF MAE:", round(base_mae, 3))
if bag_meta["ab_only_oof_mae"] is not None:
    print(f"  AB-only best OOF MAE: {bag_meta['ab_only_oof_mae']:.3f} (Code31 equivalent)")
    print(f"  ABC gain over AB: {bag_meta['ab_only_oof_mae'] - base_mae:+.3f}")
print("  OOF quantiles:", qdict(oof_base, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)))
print("  TEST quantiles:", qdict(test_base, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)))

# chronic shift
chronic_train = train_feat["primary_chronic"].astype(str).values
chronic_test  = test_feat["primary_chronic"].astype(str).values

shift_meta = tune_chronic_shift_cv(y, oof_base, chronic_train)
print("\n[best chronic shift]", shift_meta)

apply_shift = (shift_meta["cv_gain_vs_base"] >= CFG.MIN_IMPROVE_TO_APPLY) and (shift_meta["cf"] > 0)
if apply_shift:
    shifts = fit_chronic_shifts(y, oof_base, chronic_train, shift_meta["cf"])
    oof_shift = apply_chronic_shifts(oof_base, chronic_train, shifts)
    test_shift = apply_chronic_shifts(test_base, chronic_test, shifts)
    print("[apply chronic shift] YES |", {k: round(v,3) for k,v in shifts.items()})
else:
    oof_shift = oof_base.copy()
    test_shift = test_base.copy()
    shifts = {}
    print("[apply chronic shift] NO")

mae_shift = float(mean_absolute_error(y, oof_shift))

# baseline shrink
baseline_tr = train_feat["baseline_next3y"].values.astype(float)
baseline_te = test_feat["baseline_next3y"].values.astype(float)

oof_final = oof_shift.copy()
test_final = test_shift.copy()
shrink_meta = {"applied": False}

if CFG.USE_BASELINE_SHRINK:
    print("\n[baseline shrink] tuning w_model in pred = w*model + (1-w)*baseline_next3y ...")
    sm = tune_baseline_shrink(oof_shift, y, baseline_tr)
    best_w = sm["chosen_w_model"]
    oof_try = apply_baseline_shrink(oof_shift, baseline_tr, best_w)
    mae_try = float(mean_absolute_error(y, oof_try))
    gain = mae_shift - mae_try
    print("  shrink meta:", sm)
    print(f"  shrink OOF MAE: {mae_try:.3f} (gain vs pre-shrink {gain:+.3f})")

    if gain >= CFG.MIN_GAIN_TO_APPLY:
        oof_final = oof_try
        test_final = apply_baseline_shrink(test_shift, baseline_te, best_w)
        shrink_meta = {"applied": True, "w_model": float(best_w), "oof_mae": float(mae_try), "gain": float(gain), "detail": sm}
        print(f"[baseline shrink] APPLY | w_model={best_w:.2f}")
    else:
        shrink_meta = {"applied": False, "gain": float(gain), "detail": sm}
        print("[baseline shrink] SKIP (gain too small)")

# clip safety
y_max = float(np.max(y))
oof_final = np.clip(oof_final, 0.0, y_max * 1.5)
test_final = np.clip(test_final, 0.0, y_max * 1.5)

final_mae = float(mean_absolute_error(y, oof_final))

print("\n" + "="*100)
print("[FINAL SUMMARY - CODE 43]")
print(f"  SINGLE CHANGE: Added Model C (depth=3, l2=8, min_leaf=40, rs=3.0)")
print(f"  base OOF MAE (ABC bag):    {base_mae:.3f}")
print(f"  after chronic shift MAE:   {mae_shift:.3f}  (delta={mae_shift-base_mae:+.3f})")
print(f"  final OOF MAE:             {final_mae:.3f}  (delta={final_mae-base_mae:+.3f})")
print("  weight meta:", {k:v for k,v in wmeta.items() if k != 'best_ab_only'})
print("  AB-only reference:", wmeta.get("best_ab_only"))
print("  chronic shift meta:", shift_meta, "| applied:", apply_shift)
print("  baseline shrink meta:", {k:v for k,v in shrink_meta.items() if k!='detail'})
print("  OOF quantiles:", qdict(oof_final, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)))
print("  TEST quantiles:", qdict(test_final, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)))
print()
print("  >>> COMPARE TO Code31: base OOF=427.928, final OOF=427.480, LB=448.0793")
print("  >>> If ABC bag OOF < 427.9 AND ABC gain over AB > 0 -> Model C is helping.")
print("  >>> Runtime should be ~50% more than Code31 (~150s). If >5min, revert.")
print("="*100)

# write submission
sub = pd.DataFrame({
    "patient_id": test[ID_COL].values.astype(int),
    "ed_cost_next3y_usd": np.round(test_final.astype(float), 2)
})
sub = sub[["patient_id", "ed_cost_next3y_usd"]]

Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
sub.to_csv(OUT_SUB_PATH, index=False)

print("\n[SUBMISSION sanity checks]")
print("  path:", OUT_SUB_PATH)
print("  shape:", sub.shape)
print("  cols:", list(sub.columns))
print("  any NaNs:", bool(np.isnan(sub["ed_cost_next3y_usd"]).any()))
print("  min/median/max:", float(sub["ed_cost_next3y_usd"].min()), float(sub["ed_cost_next3y_usd"].median()), float(sub["ed_cost_next3y_usd"].max()))
print("  quantiles:", qdict(sub["ed_cost_next3y_usd"].values))
print(f"\n[done] total wall time: {time.time()-t0:.1f}s")
print("\nPaste back: leaderboard MAE + (base_mae, final_mae) + chosen weights (wA,wB,wC) + shift_meta + baseline_shrink_meta.")
print("KEY COMPARISON: Code31 base_OOF=427.928 | Code31 final_OOF=427.480 | Code31 LB=448.0793")

CODE 43 | Code31 + SINGLE CHANGE: Add Model C (depth=3, l2=8, min_leaf=40, rs=3.0) to ensemble
[cfg] FAST_MODE=True | folds=7 seeds=3 trim_k=0 | task=CPU
[paths] DATA_DIR=D:\AgentDs\agent_ds_healthcare

[load] reading CSVs...
[ed_cost_train] shape=(2000, 5) | cols=5 | mem=0.17 MB
[ed_cost_test] shape=(2000, 4) | cols=4 | mem=0.15 MB
[patients] shape=(4000, 5) | cols=5 | mem=0.49 MB

[target stats]
count     2000.00000
mean      3908.25191
std       1822.40192
min        306.88000
25%       2548.76750
50%       3569.09500
75%       4956.42250
max      11184.61000

[admissions] building aggregates...
  admissions features: ((4000, 4), ['patient_id', 'charlson_max', 'charlson_mean', 'pct_emergent'])

[receipts] building receipt features...
  rcpt_df shape: (4000, 45) | n_features=44

[features] building train/test feature frames...
  Receipt rcpt_sum_items vs prior_ed_cost_5y_usd match_rate(train): 1.0000
  Receipt rcpt_sum_items vs prior_ed_cost_5y_usd match_rate(test):  1.0000

[feature

In [14]:
# ============================================
# CODE 44 | Based on Code43 (which beat Code31)
# SINGLE CHANGE: Add Model D = Ridge regression on pruned features
#   - Maximally different inductive bias from CatBoost trees
#   - Near-zero runtime cost
#   - Alpha tuned via CV within each fold (nested CV)
#   - 4-way weight search (A,B,C,D) on simplex
# Code43 base: LB=447.9542, OOF=427.118
# Code31 base: LB=448.0793, OOF=427.480
# ============================================

import os, re, sys, gc, math, time, random, zlib, warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None
pd.set_option("display.max_columns", 260)
pd.set_option("display.width", 220)

# -----------------------------
# Config
# -----------------------------
class CFG:
    FAST_MODE = True
    N_FOLDS = 7
    N_SEEDS = 3 if FAST_MODE else 5
    TRIM_K = 0 if N_SEEDS < 5 else 1

    ITERS = 3200
    ES_ROUNDS = 120
    LR = 0.03
    RSM = 0.80
    SUBSAMPLE = 0.80

    USE_FULLFIT_BLEND = True
    FULLFIT_BLEND_W = 0.35

    # 4-way weight search
    W_STEP = 0.05
    STD_PEN = 0.20
    ONE_SE_TOL = 0.08

    SHIFT_FOLDS = 5
    CHRONIC_FACTORS = [0.0, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75]
    K_CHRONIC = 250.0
    CAP_CHRONIC = 250.0
    PEN_CHRONIC_ON = 0.02
    MIN_IMPROVE_TO_APPLY = 0.05

    USE_BASELINE_SHRINK = True
    SHRINK_GRID_STEP = 0.05
    SHRINK_ONESE_TOL = 0.20
    MIN_GAIN_TO_APPLY = 0.05

    TASK_TYPE = "CPU"
    GPU_BORDER_COUNT = 128

    REQUIRE_RECEIPTS = True
    MIN_RCPT_FEATURES = 20
    RCPT_MATCH_TOL = 1e-3
    RCPT_MATCH_RATE_MIN = 0.99

    # Ridge alphas to try (nested CV inside each fold)
    RIDGE_ALPHAS = [0.1, 1.0, 10.0, 50.0, 100.0, 500.0, 1000.0, 5000.0]

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

ID_COL = "patient_id"
TARGET = "ed_cost_next3y_usd"

DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"
TRAIN_PATH = os.path.join(DATA_DIR, "ed_cost_train.csv")
TEST_PATH  = os.path.join(DATA_DIR, "ed_cost_test.csv")
PATIENTS_PATH = os.path.join(DATA_DIR, "patients.csv")
ADM_TRAIN_PATH = os.path.join(DATA_DIR, "admissions_train.csv")
ADM_TEST_PATH  = os.path.join(DATA_DIR, "admissions_test.csv")
RECEIPTS_JOBLIB_PATH = os.path.join(DATA_DIR, "receipts_parsed.joblib")
OUT_SUB_PATH = os.path.join(DATA_DIR, "submission.csv")

# -----------------------------
# Minimal deps
# -----------------------------
def _pip_install(pkg: str):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    from joblib import load as joblib_load
except Exception:
    _pip_install("joblib")
    from joblib import load as joblib_load

try:
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    from sklearn.metrics import mean_absolute_error
    from sklearn.linear_model import Ridge
except Exception:
    _pip_install("scikit-learn")
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    from sklearn.metrics import mean_absolute_error
    from sklearn.linear_model import Ridge

try:
    from catboost import CatBoostRegressor
except Exception:
    _pip_install("catboost")
    from catboost import CatBoostRegressor

# -----------------------------
# Utilities (IDENTICAL to Code31/43)
# -----------------------------
def must_exist(path: str, name: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {name}: {path}")

def log_shape(name: str, df: pd.DataFrame):
    mem = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"[{name}] shape={df.shape} | cols={df.shape[1]} | mem={mem:.2f} MB")

def qdict(x, qs=(0,0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99,1.0)):
    x = np.asarray(x, dtype=float)
    return {q: float(np.quantile(x, q)) for q in qs}

def standardize_zip3(z):
    if z is None or (isinstance(z, float) and np.isnan(z)):
        return None
    s = re.sub(r"\D", "", str(z).strip())
    if not s:
        return None
    return s.zfill(3)

def norm_code(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip().upper()
    if s == "" or s.lower() == "nan":
        return None
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    s = re.sub(r"\s+", "", s)
    return s

def safe_num_series(s: pd.Series, default=0.0) -> pd.Series:
    v = pd.to_numeric(s, errors="coerce")
    v = v.replace([np.inf, -np.inf], np.nan).fillna(default)
    return v

def stable_hash(s: str) -> int:
    return int(zlib.crc32(s.encode("utf-8")) & 0xffffffff)

def gini(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if x.size == 0:
        return 0.0
    if np.all(x == 0):
        return 0.0
    x = np.sort(x)
    n = x.size
    cumx = np.cumsum(x)
    return float((n + 1 - 2*np.sum(cumx)/cumx[-1]) / n)

def trimmed_mean(mat: np.ndarray, trim_k: int = 1) -> np.ndarray:
    mat = np.asarray(mat, dtype=float)
    if mat.ndim != 2:
        raise ValueError("trimmed_mean expects 2D array")
    n = mat.shape[0]
    if trim_k <= 0 or n < 2*trim_k + 1:
        return np.mean(mat, axis=0)
    s = np.sort(mat, axis=0)
    return np.mean(s[trim_k:n-trim_k, :], axis=0)

# -----------------------------
# Admissions aggregates (IDENTICAL)
# -----------------------------
def load_admissions_features(adm_train_path: str, adm_test_path: str) -> Optional[pd.DataFrame]:
    dfs = []
    for path in [adm_train_path, adm_test_path]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            if "readmit_30d" in df.columns:
                df = df.drop(columns=["readmit_30d"], errors="ignore")
            dfs.append(df)
    if not dfs:
        return None

    adm = pd.concat(dfs, ignore_index=True)
    need = ["patient_id","charlson_band","acuity_emergent"]
    if not all(c in adm.columns for c in need):
        return None

    adm["patient_id"] = pd.to_numeric(adm["patient_id"], errors="coerce").astype("Int64")
    adm = adm.dropna(subset=["patient_id"]).copy()
    adm["patient_id"] = adm["patient_id"].astype(int)
    adm["charlson_band"] = pd.to_numeric(adm["charlson_band"], errors="coerce")
    adm["acuity_emergent"] = pd.to_numeric(adm["acuity_emergent"], errors="coerce")

    out = adm.groupby("patient_id").agg(
        charlson_max=("charlson_band","max"),
        charlson_mean=("charlson_band","mean"),
        pct_emergent=("acuity_emergent","mean"),
    ).reset_index()

    for c in ["charlson_max","charlson_mean","pct_emergent"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0.0)
    return out

# -----------------------------
# Receipts features (IDENTICAL)
# -----------------------------
def build_receipt_features_from_lineitems(li: pd.DataFrame) -> pd.DataFrame:
    li = li.copy()

    code_col = None
    for c in ["code","cpt","cpt_code","hcpcs","proc_code","procedure_code"]:
        if c in li.columns:
            code_col = c; break

    total_col = None
    for c in ["line_total","line_total_usd","total","amount","line_cost","sum_items","item_total","extended_price","sum_line_total"]:
        if c in li.columns:
            total_col = c; break

    unit_col = None
    for c in ["unit_price","unitprice","unit_cost","unitcost"]:
        if c in li.columns:
            unit_col = c; break

    if "patient_id" not in li.columns or code_col is None or total_col is None:
        raise ValueError("receipts lineitems missing required columns.")

    li["patient_id"] = pd.to_numeric(li["patient_id"], errors="coerce").astype("Int64")
    li = li.dropna(subset=["patient_id"]).copy()
    li["patient_id"] = li["patient_id"].astype(int)
    li["code"] = li[code_col].map(norm_code)
    li = li.dropna(subset=["code"]).copy()
    li["amt"] = pd.to_numeric(li[total_col], errors="coerce").fillna(0.0).astype(float)
    li.loc[li["amt"] < 0, "amt"] = 0.0

    if unit_col is not None:
        li["unit_price"] = pd.to_numeric(li[unit_col], errors="coerce").replace([np.inf,-np.inf], np.nan)
    else:
        li["unit_price"] = np.nan

    rcpt_sum = li.groupby("patient_id")["amt"].sum().rename("rcpt_sum_items")
    li = li.join(rcpt_sum, on="patient_id")
    denom = li["rcpt_sum_items"].replace(0.0, np.nan)
    share = (li["amt"] / denom).replace([np.inf,-np.inf], np.nan).fillna(0.0)

    cost_hhi = (share * share).groupby(li["patient_id"]).sum().rename("cost_hhi")
    top1_share = share.groupby(li["patient_id"]).max().rename("top1_share")

    def _topk_sum(s, k=3):
        a = np.sort(s.values.astype(float))[::-1]
        return float(a[:k].sum()) if a.size else 0.0
    top3_share = share.groupby(li["patient_id"]).apply(lambda s: _topk_sum(s, 3)).rename("top3_share")

    gini_amt = li.groupby("patient_id")["amt"].apply(lambda s: gini(s.values)).rename("gini_line_total")
    max_line_total = li.groupby("patient_id")["amt"].max().rename("max_line_total")

    code_num = pd.to_numeric(li["code"].where(li["code"].str.fullmatch(r"\d+"), None), errors="coerce")

    em_codes = ["99281","99282","99283","99284","99285"]
    is_em = li["code"].isin(em_codes)
    em_map = {"99281":1,"99282":2,"99283":3,"99284":4,"99285":5}
    em_level = li["code"].map(em_map).fillna(0).astype(float)
    is_crit = li["code"].isin(["99291","99292"])
    is_obs = li["code"].str.startswith("G037", na=False)
    is_high = li["code"].isin(["31500","36556","32551","36620","92950"])
    is_lab = code_num.between(80000, 89999)
    is_imaging = code_num.between(70000, 79999)
    is_proc_general = code_num.between(10000, 69999)
    is_proc_any = is_high | (is_proc_general & (~is_high) & (~is_em) & (~is_crit))

    n_unique_codes = li.groupby("patient_id")["code"].nunique().rename("n_unique_codes")
    n_em_codes = is_em.astype(int).groupby(li["patient_id"]).sum().rename("n_em_codes")
    max_em_level = em_level.groupby(li["patient_id"]).max().rename("max_em_level")
    sum_em_level = (em_level * is_em.astype(int)).groupby(li["patient_id"]).sum().rename("sum_em_level")
    avg_em_level = (sum_em_level / n_em_codes.replace(0, np.nan)).fillna(0.0).rename("avg_em_level")
    n_high_em = ((em_level >= 4) & is_em).astype(int).groupby(li["patient_id"]).sum().rename("n_high_em")

    em_total = li.loc[is_em].groupby("patient_id")["amt"].sum().rename("em_total")
    crit_total = li.loc[is_crit].groupby("patient_id")["amt"].sum().rename("crit_total")
    proc_total = li.loc[is_proc_any].groupby("patient_id")["amt"].sum().rename("proc_total")
    img_total = li.loc[is_imaging].groupby("patient_id")["amt"].sum().rename("img_total")
    lab_total = li.loc[is_lab].groupby("patient_id")["amt"].sum().rename("lab_total")
    high_total = li.loc[is_high].groupby("patient_id")["amt"].sum().rename("high_total")

    n_procedures = is_proc_any.astype(int).groupby(li["patient_id"]).sum().rename("n_procedures")
    n_imaging = is_imaging.astype(int).groupby(li["patient_id"]).sum().rename("n_imaging")
    n_lab = is_lab.astype(int).groupby(li["patient_id"]).sum().rename("n_lab")

    has_critical_care = is_crit.astype(int).groupby(li["patient_id"]).max().rename("has_critical_care")
    has_high_acuity = is_high.astype(int).groupby(li["patient_id"]).max().rename("has_high_acuity")
    has_observation = is_obs.astype(int).groupby(li["patient_id"]).max().rename("has_observation")
    has_imaging = is_imaging.astype(int).groupby(li["patient_id"]).max().rename("has_imaging")

    def has_code(code, name):
        return (li["code"].eq(code).astype(int).groupby(li["patient_id"]).max()).rename(name)

    has_intub_31500 = has_code("31500","has_intub_31500")
    has_cvc_36556 = has_code("36556","has_cvc_36556")
    has_cpr_92950 = has_code("92950","has_cpr_92950")
    has_artline_36620 = has_code("36620","has_artline_36620")
    has_ct_head_70450 = has_code("70450","has_ct_head_70450")
    has_99285 = has_code("99285","has_99285")
    has_ct_abdpel_74177 = has_code("74177","has_ct_abdpel_74177")
    has_troponin_84484 = has_code("84484","has_troponin_84484")
    has_obs_G0378 = has_code("G0378","has_obs_G0378")

    med_unit_all = li.loc[li["unit_price"].notna()].groupby("patient_id")["unit_price"].median().rename("median_unit_price")

    def _median_unit(mask, name):
        sub = li.loc[mask & li["unit_price"].notna(), ["patient_id","unit_price"]]
        if sub.empty: return None
        return sub.groupby("patient_id")["unit_price"].median().rename(name)

    med_unit_em  = _median_unit(is_em, "median_unit_price_em")
    med_unit_img = _median_unit(is_imaging, "median_unit_price_imaging")
    med_unit_lab = _median_unit(is_lab, "median_unit_price_lab")

    out = pd.concat([
        rcpt_sum, n_unique_codes, cost_hhi, top1_share, top3_share, gini_amt, max_line_total,
        n_em_codes, max_em_level, avg_em_level, n_high_em,
        has_critical_care, has_high_acuity, has_observation, has_imaging,
        has_intub_31500, has_cvc_36556, has_cpr_92950, has_artline_36620,
        has_ct_head_70450, has_99285, has_ct_abdpel_74177, has_troponin_84484, has_obs_G0378,
        n_procedures, n_imaging, n_lab
    ], axis=1).reset_index()

    for s in [em_total, crit_total, proc_total, img_total, lab_total, high_total]:
        out = out.merge(s.reset_index(), on="patient_id", how="left")

    for c in ["em_total","crit_total","proc_total","img_total","lab_total","high_total","rcpt_sum_items"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    denom2 = out["rcpt_sum_items"].replace(0.0, np.nan)
    out["pct_cost_em"] = (out["em_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_procedure"] = (out["proc_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_critical"] = (out["crit_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_imaging"] = (out["img_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_lab"] = (out["lab_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_high_acuity"] = (out["high_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)

    out["cost_per_em"] = np.where(out["n_em_codes"] > 0,
                                  out["rcpt_sum_items"] / out["n_em_codes"].clip(lower=1),
                                  out["rcpt_sum_items"])
    out["n_high_acuity_total"] = (
        out["has_intub_31500"].fillna(0).astype(int) + out["has_cvc_36556"].fillna(0).astype(int)
        + out["has_cpr_92950"].fillna(0).astype(int) + out["has_artline_36620"].fillna(0).astype(int)
        + out["has_critical_care"].fillna(0).astype(int)
    ).astype(int)

    out = out.merge(med_unit_all.reset_index(), on="patient_id", how="left")
    if med_unit_em is not None:  out = out.merge(med_unit_em.reset_index(), on="patient_id", how="left")
    else: out["median_unit_price_em"] = np.nan
    if med_unit_img is not None: out = out.merge(med_unit_img.reset_index(), on="patient_id", how="left")
    else: out["median_unit_price_imaging"] = np.nan
    if med_unit_lab is not None: out = out.merge(med_unit_lab.reset_index(), on="patient_id", how="left")
    else: out["median_unit_price_lab"] = np.nan

    for c in ["median_unit_price","median_unit_price_em","median_unit_price_imaging","median_unit_price_lab","max_line_total"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0.0)
        out["log1p_" + c] = np.log1p(out[c].clip(lower=0.0))

    out.drop(columns=[c for c in ["em_total","crit_total","proc_total","img_total","lab_total","high_total"] if c in out.columns],
             inplace=True, errors="ignore")

    for c in out.columns:
        if c == "patient_id": continue
        out[c] = pd.to_numeric(out[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0.0)
    return out

def load_receipts_joblib(joblib_path: str) -> Optional[pd.DataFrame]:
    if not os.path.exists(joblib_path):
        return None
    data = joblib_load(joblib_path)

    if isinstance(data, pd.DataFrame):
        df = data
        if ("patient_id" in df.columns) and any(c in df.columns for c in ["code","cpt","cpt_code","hcpcs","proc_code","procedure_code"]):
            try: return build_receipt_features_from_lineitems(df)
            except Exception: pass
        if "patient_id" in df.columns: return df
        return None

    if isinstance(data, dict):
        cand_keys = ["lineitems_df","lineitems","items_df","items","line_items_df","line_items","df","data"]
        for k in cand_keys:
            if k in data and isinstance(data[k], pd.DataFrame):
                df = data[k]
                if ("patient_id" in df.columns) and any(c in df.columns for c in ["code","cpt","cpt_code","hcpcs","proc_code","procedure_code"]):
                    return build_receipt_features_from_lineitems(df)
                if "patient_id" in df.columns: return df
        for v in data.values():
            if isinstance(v, pd.DataFrame) and "patient_id" in v.columns:
                df = v
                if any(c in df.columns for c in ["code","cpt","cpt_code","hcpcs","proc_code","procedure_code"]):
                    return build_receipt_features_from_lineitems(df)
                return df
        return None

    if isinstance(data, (list, tuple)):
        dfs = [x for x in data if isinstance(x, pd.DataFrame)]
        for df in dfs:
            if ("patient_id" in df.columns) and any(c in df.columns for c in ["code","cpt","cpt_code","hcpcs","proc_code","procedure_code"]):
                return build_receipt_features_from_lineitems(df)
        for df in dfs:
            if "patient_id" in df.columns: return df
        return None
    return None

# -----------------------------
# Feature engineering (IDENTICAL)
# -----------------------------
def build_features(ed_df, patients_df, adm_df, rcpt_df):
    feat = ed_df.copy()
    chronic_map = {"PNEUMONIA":0, "DIABETESCOMP":1, "HF":2}
    feat["primary_chronic"] = feat["primary_chronic"].astype(str)
    feat["chronic_encoded"] = feat["primary_chronic"].str.upper().map(chronic_map).fillna(-1).astype(float)

    feat["prior_ed_visits_5y"] = safe_num_series(feat["prior_ed_visits_5y"], default=0.0).clip(lower=0.0)
    feat["prior_ed_cost_5y_usd"] = safe_num_series(feat["prior_ed_cost_5y_usd"], default=0.0).clip(lower=0.0)

    feat["prior_cost_cap20k"] = feat["prior_ed_cost_5y_usd"].clip(upper=20000.0)
    feat["sqrt_prior_cost"] = np.sqrt(feat["prior_ed_cost_5y_usd"].clip(lower=0.0))
    feat["log_prior_cost"] = np.log1p(feat["prior_ed_cost_5y_usd"].clip(lower=0.0))
    feat["log_prior_cost_cap20k"] = np.log1p(feat["prior_cost_cap20k"].clip(lower=0.0))
    feat["log_visits"] = np.log1p(feat["prior_ed_visits_5y"].clip(lower=0.0))
    feat["cost_per_visit"] = feat["prior_ed_cost_5y_usd"] / feat["prior_ed_visits_5y"].clip(lower=1.0)
    feat["baseline_next3y"] = feat["prior_ed_cost_5y_usd"] * (3.0/5.0)

    p = patients_df.copy()
    p["patient_id"] = pd.to_numeric(p["patient_id"], errors="coerce").astype("Int64")
    p = p.dropna(subset=["patient_id"]).copy()
    p["patient_id"] = p["patient_id"].astype(int)
    p["age"] = pd.to_numeric(p["age"], errors="coerce").fillna(p["age"].median()).clip(lower=0, upper=110)
    p["sex_encoded"] = (p["sex"].astype(str).str.upper() == "M").astype(int)
    ins = p["insurance"].astype(str).str.lower()
    ins_map = {"private":2, "public":1, "self_pay":0, "selfpay":0}
    p["insurance_encoded"] = ins.map(ins_map).fillna(-1).astype(float)
    z3 = p["zip3"].apply(standardize_zip3).astype("string")
    zr = z3.fillna("000").str.replace(r"\D","", regex=True).str.zfill(3).str[0]
    p["zip_region"] = pd.to_numeric(zr, errors="coerce").fillna(-1).astype(float)

    feat = feat.merge(p[["patient_id","age","sex_encoded","insurance_encoded","zip_region"]], on="patient_id", how="left")
    feat["ins_x_chronic"] = feat["insurance_encoded"].fillna(-1) * feat["chronic_encoded"].fillna(-1)

    if adm_df is not None:
        feat = feat.merge(adm_df, on="patient_id", how="left")
        for c in ["charlson_max","charlson_mean","pct_emergent"]:
            if c in feat.columns:
                feat[c] = pd.to_numeric(feat[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0.0)

    if rcpt_df is not None:
        feat = feat.merge(rcpt_df, on="patient_id", how="left")
        for c in rcpt_df.columns:
            if c == "patient_id": continue
            feat[c] = pd.to_numeric(feat[c], errors="coerce").replace([np.inf,-np.inf], np.nan)
            if feat[c].isna().any():
                feat[c] = feat[c].fillna(feat[c].median() if not feat[c].isna().all() else 0.0)

    if "pct_cost_critical" in feat.columns:
        feat["logprior_x_pctcritical"] = feat["log_prior_cost"] * feat["pct_cost_critical"]
    if "n_high_acuity_total" in feat.columns:
        feat["logprior_x_highacu"] = feat["log_prior_cost"] * feat["n_high_acuity_total"]
    if "n_unique_codes" in feat.columns:
        feat["cost_per_code"] = feat["prior_ed_cost_5y_usd"] / feat["n_unique_codes"].clip(lower=1.0)

    feat.replace([np.inf, -np.inf], np.nan, inplace=True)
    for c in feat.columns:
        if c in ["patient_id", "primary_chronic", TARGET]: continue
        feat[c] = pd.to_numeric(feat[c], errors="coerce")
        if feat[c].isna().any():
            feat[c] = feat[c].fillna(feat[c].median() if not feat[c].isna().all() else 0.0)
    return feat

def get_numeric_feature_cols(df):
    exclude = {"patient_id","primary_chronic",TARGET,"sex","insurance","zip3"}
    return [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]

def drop_constants(cols, df):
    return [c for c in cols if c in df.columns and df[c].nunique(dropna=False) > 1]

# -----------------------------
# Training CatBoost A/B/C (IDENTICAL to Code43)
# -----------------------------
def _adjust_params_for_task(params):
    p = dict(params)
    if CFG.TASK_TYPE.upper() == "GPU":
        p.pop("rsm", None)
        p["border_count"] = CFG.GPU_BORDER_COUNT
    return p

def train_catboost_models(train_feat, test_feat, feat_full, feat_pruned, strat_vec):
    """Train CatBoost A/B/C exactly as Code43."""
    y = train_feat[TARGET].values.astype(float)

    model_specs = {
        "A_full_d5": dict(
            cols=feat_full,
            params=dict(
                loss_function="RMSE", eval_metric="MAE",
                depth=5, l2_leaf_reg=5, min_data_in_leaf=28,
                learning_rate=CFG.LR, iterations=CFG.ITERS,
                rsm=CFG.RSM, bootstrap_type="Bernoulli", subsample=CFG.SUBSAMPLE,
                random_strength=1.0,
            )
        ),
        "B_pruned_d4": dict(
            cols=feat_pruned,
            params=dict(
                loss_function="RMSE", eval_metric="MAE",
                depth=4, l2_leaf_reg=4, min_data_in_leaf=32,
                learning_rate=CFG.LR, iterations=CFG.ITERS,
                rsm=CFG.RSM, bootstrap_type="Bernoulli", subsample=CFG.SUBSAMPLE,
                random_strength=2.0,
            )
        ),
        "C_pruned_d3": dict(
            cols=feat_pruned,
            params=dict(
                loss_function="RMSE", eval_metric="MAE",
                depth=3, l2_leaf_reg=8, min_data_in_leaf=40,
                learning_rate=CFG.LR, iterations=CFG.ITERS,
                rsm=0.70, bootstrap_type="Bernoulli", subsample=0.75,
                random_strength=3.0,
            )
        ),
    }

    oof_by_seed = {m: [] for m in model_specs}
    test_by_seed = {m: [] for m in model_specs}
    best_iters = {m: [] for m in model_specs}

    print(f"\n[CatBoost training] {CFG.TASK_TYPE} | folds={CFG.N_FOLDS} seeds={CFG.N_SEEDS} | iters={CFG.ITERS} es={CFG.ES_ROUNDS}")
    print("Models:", list(model_specs.keys()), "| TRIM_K:", CFG.TRIM_K)

    t_all = time.time()
    for sidx in range(CFG.N_SEEDS):
        rs = SEED + sidx * 7
        kf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=rs)

        oof_seed = {m: np.zeros(len(train_feat), dtype=float) for m in model_specs}
        test_seed_foldbag = {m: np.zeros(len(test_feat), dtype=float) for m in model_specs}
        best_iters_seed = {m: [] for m in model_specs}

        for fold, (ti, vi) in enumerate(kf.split(train_feat, strat_vec), 1):
            for mname, spec in model_specs.items():
                cols = spec["cols"]
                params = _adjust_params_for_task(spec["params"])

                cb = CatBoostRegressor(
                    **params, task_type=CFG.TASK_TYPE, thread_count=-1, verbose=0,
                    allow_writing_files=False,
                    random_seed=int(rs + fold * 31 + stable_hash(mname) % 997),
                    early_stopping_rounds=CFG.ES_ROUNDS,
                )
                cb.fit(train_feat[cols].iloc[ti], y[ti],
                       eval_set=(train_feat[cols].iloc[vi], y[vi]), verbose=0)

                try: bi = int(cb.get_best_iteration())
                except: bi = None
                if bi is not None and bi > 0:
                    best_iters[mname].append(bi)
                    best_iters_seed[mname].append(bi)

                oof_seed[mname][vi] = np.clip(cb.predict(train_feat[cols].iloc[vi]).astype(float), 0, None)
                test_seed_foldbag[mname] += np.clip(cb.predict(test_feat[cols]).astype(float), 0, None) / CFG.N_FOLDS
                del cb; gc.collect()

        test_seed_final = {}
        if CFG.USE_FULLFIT_BLEND:
            for mname, spec in model_specs.items():
                cols = spec["cols"]
                params = _adjust_params_for_task(spec["params"])
                it_med = int(np.median(best_iters_seed[mname])) if best_iters_seed[mname] else int(CFG.ITERS * 0.45)
                params["iterations"] = int(max(350, min(CFG.ITERS, it_med + 150)))
                cb_full = CatBoostRegressor(
                    **params, task_type=CFG.TASK_TYPE, thread_count=-1, verbose=0,
                    allow_writing_files=False,
                    random_seed=int(rs + 999 + stable_hash("FULL_"+mname) % 997),
                )
                cb_full.fit(train_feat[cols], y, verbose=0)
                pred_full = np.clip(cb_full.predict(test_feat[cols]).astype(float), 0, None)
                test_seed_final[mname] = (1-CFG.FULLFIT_BLEND_W)*test_seed_foldbag[mname] + CFG.FULLFIT_BLEND_W*pred_full
                del cb_full; gc.collect()
        else:
            test_seed_final = test_seed_foldbag

        seed_maes = {m: float(mean_absolute_error(y, oof_seed[m])) for m in model_specs}
        print(f"  Seed {sidx+1}/{CFG.N_SEEDS} OOF MAE: " + " | ".join([f"{m}={seed_maes[m]:.2f}" for m in model_specs]))
        for m in model_specs:
            oof_by_seed[m].append(oof_seed[m])
            test_by_seed[m].append(test_seed_final[m])

    print(f"\n[CatBoost training] total time: {time.time()-t_all:.1f}s")
    for m in model_specs:
        avg_oof = trimmed_mean(np.vstack(oof_by_seed[m]), trim_k=CFG.TRIM_K)
        print(f"  {m:15s} seed-agg OOF MAE: {mean_absolute_error(y, avg_oof):.2f}")

    return oof_by_seed, test_by_seed

# =============================================
# NEW: Model D = Ridge regression (single seed, same folds as seed 0)
# =============================================
def train_ridge_model(train_feat, test_feat, feat_cols, strat_vec):
    """Train Ridge regression with StandardScaler. Uses RidgeCV-like approach
    within each fold to select alpha, then predicts OOF and test."""
    y = train_feat[TARGET].values.astype(float)
    n_train = len(train_feat)
    n_test = len(test_feat)

    print(f"\n[Ridge training] features={len(feat_cols)} | folds={CFG.N_FOLDS} seeds={CFG.N_SEEDS}")
    print(f"  alphas to try: {CFG.RIDGE_ALPHAS}")

    t0 = time.time()

    oof_by_seed = []
    test_by_seed = []

    for sidx in range(CFG.N_SEEDS):
        rs = SEED + sidx * 7
        kf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=rs)

        oof = np.zeros(n_train, dtype=float)
        test_preds_folds = np.zeros(n_test, dtype=float)
        chosen_alphas = []

        for fold, (ti, vi) in enumerate(kf.split(train_feat, strat_vec), 1):
            X_tr = train_feat[feat_cols].iloc[ti].values.astype(float)
            y_tr = y[ti]
            X_va = train_feat[feat_cols].iloc[vi].values.astype(float)
            y_va = y[vi]
            X_te = test_feat[feat_cols].values.astype(float)

            # Scale
            scaler = StandardScaler()
            X_tr_s = scaler.fit_transform(X_tr)
            X_va_s = scaler.transform(X_va)
            X_te_s = scaler.transform(X_te)

            # Inner CV to pick alpha (simple: use val set MAE)
            best_alpha = CFG.RIDGE_ALPHAS[0]
            best_val_mae = 1e18
            for alpha in CFG.RIDGE_ALPHAS:
                m = Ridge(alpha=alpha, fit_intercept=True)
                m.fit(X_tr_s, y_tr)
                pred_va = m.predict(X_va_s)
                val_mae = float(mean_absolute_error(y_va, pred_va))
                if val_mae < best_val_mae:
                    best_val_mae = val_mae
                    best_alpha = alpha

            chosen_alphas.append(best_alpha)

            # Refit with best alpha
            m = Ridge(alpha=best_alpha, fit_intercept=True)
            m.fit(X_tr_s, y_tr)
            oof[vi] = np.clip(m.predict(X_va_s), 0, None)
            test_preds_folds += np.clip(m.predict(X_te_s), 0, None) / CFG.N_FOLDS

        # Fullfit blend for test
        if CFG.USE_FULLFIT_BLEND:
            # Use most common alpha from CV
            from collections import Counter
            alpha_counts = Counter(chosen_alphas)
            best_global_alpha = alpha_counts.most_common(1)[0][0]

            scaler_full = StandardScaler()
            X_all_s = scaler_full.fit_transform(train_feat[feat_cols].values.astype(float))
            X_te_s = scaler_full.transform(test_feat[feat_cols].values.astype(float))
            m_full = Ridge(alpha=best_global_alpha, fit_intercept=True)
            m_full.fit(X_all_s, y)
            test_full = np.clip(m_full.predict(X_te_s), 0, None)
            test_final = (1-CFG.FULLFIT_BLEND_W)*test_preds_folds + CFG.FULLFIT_BLEND_W*test_full
        else:
            test_final = test_preds_folds
            best_global_alpha = None

        oof_mae = float(mean_absolute_error(y, oof))
        print(f"  Seed {sidx+1}/{CFG.N_SEEDS} Ridge OOF MAE: {oof_mae:.2f} | alphas chosen: {chosen_alphas} | global: {best_global_alpha}")

        oof_by_seed.append(oof)
        test_by_seed.append(test_final)

    # Aggregate seeds
    oof_avg = trimmed_mean(np.vstack(oof_by_seed), trim_k=CFG.TRIM_K)
    test_avg = trimmed_mean(np.vstack(test_by_seed), trim_k=CFG.TRIM_K)
    ridge_oof_mae = float(mean_absolute_error(y, oof_avg))

    print(f"  Ridge seed-agg OOF MAE: {ridge_oof_mae:.2f}")
    print(f"  Ridge total time: {time.time()-t0:.1f}s")

    return oof_by_seed, test_by_seed

# =============================================
# 4-way weight search on (A,B,C,D) simplex
# =============================================
def weight_search_4way(oof_by_seed, y):
    """Search over (wA, wB, wC, wD) simplex where sum=1, step=0.05"""
    step = CFG.W_STEP
    grid = np.round(np.arange(0.0, 1.0 + 1e-9, step), 10)

    model_names = ["A_full_d5", "B_pruned_d4", "C_pruned_d3", "D_ridge"]
    rows = []

    for wA in grid:
        for wB in grid:
            if wA + wB > 1.0 + 1e-9:
                break
            for wC in grid:
                wD = round(1.0 - wA - wB - wC, 10)
                if wD < -1e-9 or wD > 1.0 + 1e-9:
                    continue
                if wA + wB + wC > 1.0 + 1e-9:
                    break
                wD = max(0.0, wD)
                maes = []
                for s in range(CFG.N_SEEDS):
                    p = (wA*oof_by_seed["A_full_d5"][s]
                         + wB*oof_by_seed["B_pruned_d4"][s]
                         + wC*oof_by_seed["C_pruned_d3"][s]
                         + wD*oof_by_seed["D_ridge"][s])
                    maes.append(float(mean_absolute_error(y, p)))
                mean_m = float(np.mean(maes))
                std_m = float(np.std(maes))
                obj = mean_m + CFG.STD_PEN*std_m
                rows.append((obj, mean_m, std_m, float(wA), float(wB), float(wC), float(wD)))

    rows.sort(key=lambda x: x[0])

    # Also find ABC-only (wD=0) and AB-only (wC=wD=0)
    abc_only = [r for r in rows if abs(r[6]) < 1e-9]
    abc_only.sort(key=lambda x: x[0])
    ab_only = [r for r in rows if abs(r[5]) < 1e-9 and abs(r[6]) < 1e-9]
    ab_only.sort(key=lambda x: x[0])

    print(f"\n[4-way ensemble search] Total combos: {len(rows)} | Top 15:")
    for i, r in enumerate(rows[:15], 1):
        obj, mean_m, std_m, wA, wB, wC, wD = r
        marker = ""
        if abs(wD) < 1e-9 and abs(wC) < 1e-9: marker = " [AB-only]"
        elif abs(wD) < 1e-9: marker = " [ABC-only]"
        print(f"  #{i:02d} obj={obj:.3f} mean={mean_m:.3f} std={std_m:.3f} | wA={wA:.2f} wB={wB:.2f} wC={wC:.2f} wD={wD:.2f}{marker}")

    print(f"\n[comparison]")
    if ab_only:
        print(f"  AB-only  best: obj={ab_only[0][0]:.3f} mean={ab_only[0][1]:.3f} | wA={ab_only[0][3]:.2f} wB={ab_only[0][4]:.2f}")
    if abc_only:
        print(f"  ABC-only best: obj={abc_only[0][0]:.3f} mean={abc_only[0][1]:.3f} | wA={abc_only[0][3]:.2f} wB={abc_only[0][4]:.2f} wC={abc_only[0][5]:.2f}")
    print(f"  ABCD     best: obj={rows[0][0]:.3f} mean={rows[0][1]:.3f} | wA={rows[0][3]:.2f} wB={rows[0][4]:.2f} wC={rows[0][5]:.2f} wD={rows[0][6]:.2f}")
    if abc_only:
        print(f"  ABCD gain over ABC: {abc_only[0][0] - rows[0][0]:+.3f} obj")

    # One-SE: prefer simpler (less wD, less wC, less wA)
    best_obj = rows[0][0]
    eligible = [r for r in rows if r[0] <= best_obj + CFG.ONE_SE_TOL]
    chosen = min(eligible, key=lambda r: (r[6], r[5], r[3], r[0]))  # prefer less D, less C, less A
    ch_obj, ch_mean, ch_std, ch_wA, ch_wB, ch_wC, ch_wD = chosen

    print(f"\n[one-SE selection] wA={ch_wA:.2f} wB={ch_wB:.2f} wC={ch_wC:.2f} wD={ch_wD:.2f} | obj={ch_obj:.3f}")

    meta = {
        "best": {"wA":rows[0][3],"wB":rows[0][4],"wC":rows[0][5],"wD":rows[0][6],"obj":rows[0][0],"mean":rows[0][1]},
        "chosen": {"wA":ch_wA,"wB":ch_wB,"wC":ch_wC,"wD":ch_wD,"obj":ch_obj,"mean":ch_mean,"std":ch_std},
        "abc_best": {"obj":abc_only[0][0],"mean":abc_only[0][1]} if abc_only else None,
        "ab_best": {"obj":ab_only[0][0],"mean":ab_only[0][1]} if ab_only else None,
    }
    return meta

def build_4way_bag_preds(oof_by_seed, test_by_seed, chosen_meta, y):
    """Bag predictions around chosen 4-way weight point."""
    ch = chosen_meta["chosen"]
    wA0, wB0, wC0, wD0 = ch["wA"], ch["wB"], ch["wC"], ch["wD"]
    step = CFG.W_STEP

    # Generate neighbors on simplex within 1 step
    bag_points = set()
    for dA in [-step, 0, step]:
        for dB in [-step, 0, step]:
            for dC in [-step, 0, step]:
                wA = round(wA0 + dA, 10)
                wB = round(wB0 + dB, 10)
                wC = round(wC0 + dC, 10)
                wD = round(1.0 - wA - wB - wC, 10)
                if all(w >= -1e-9 and w <= 1+1e-9 for w in [wA,wB,wC,wD]):
                    bag_points.add((max(0,wA), max(0,wB), max(0,wC), max(0,wD)))

    bag_points = sorted(bag_points)
    print(f"\n[4-way weight-bagging] {len(bag_points)} points around chosen ({wA0:.2f},{wB0:.2f},{wC0:.2f},{wD0:.2f})")

    oof_preds = []
    test_preds = []

    for (wA, wB, wC, wD) in bag_points:
        oof_parts = []
        test_parts = []
        for s in range(CFG.N_SEEDS):
            o = (wA*oof_by_seed["A_full_d5"][s] + wB*oof_by_seed["B_pruned_d4"][s]
                 + wC*oof_by_seed["C_pruned_d3"][s] + wD*oof_by_seed["D_ridge"][s])
            t = (wA*test_by_seed["A_full_d5"][s] + wB*test_by_seed["B_pruned_d4"][s]
                 + wC*test_by_seed["C_pruned_d3"][s] + wD*test_by_seed["D_ridge"][s])
            oof_parts.append(o)
            test_parts.append(t)

        oof_trim = trimmed_mean(np.vstack(oof_parts), trim_k=CFG.TRIM_K)
        test_trim = trimmed_mean(np.vstack(test_parts), trim_k=CFG.TRIM_K)
        oof_preds.append(oof_trim)
        test_preds.append(test_trim)

    oof_bag = np.mean(np.vstack(oof_preds), axis=0)
    test_bag = np.mean(np.vstack(test_preds), axis=0)

    meta = {"n_bag_points": len(bag_points)}
    return oof_bag, test_bag, meta

# -----------------------------
# Chronic residual shift (IDENTICAL)
# -----------------------------
def _shrink(n, k):
    return float(n / (n + k)) if n > 0 else 0.0

def fit_chronic_shifts(y_tr, p_tr, chronic_tr, cf):
    resid = np.asarray(y_tr, float) - np.asarray(p_tr, float)
    chronic_tr = np.asarray(chronic_tr).astype(str)
    shifts = {}
    for g in np.unique(chronic_tr):
        m = (chronic_tr == g)
        n = int(m.sum())
        med = float(np.median(resid[m])) if n > 0 else 0.0
        val = cf * _shrink(n, CFG.K_CHRONIC) * med
        shifts[g] = float(np.clip(val, -CFG.CAP_CHRONIC, CFG.CAP_CHRONIC))
    return shifts

def apply_chronic_shifts(p, chronic, shifts):
    p = np.asarray(p, float).copy()
    chronic = np.asarray(chronic).astype(str)
    for g, sh in shifts.items():
        p[chronic == g] += sh
    return p

def tune_chronic_shift_cv(y, p_base, chronic):
    y = np.asarray(y, float)
    p_base = np.asarray(p_base, float)
    chronic = np.asarray(chronic).astype(str)
    base_mae = float(mean_absolute_error(y, p_base))
    strat = LabelEncoder().fit_transform(chronic)
    kf = StratifiedKFold(n_splits=CFG.SHIFT_FOLDS, shuffle=True, random_state=SEED)
    rows = []
    best = None
    for cf in CFG.CHRONIC_FACTORS:
        pcv = np.zeros_like(p_base, dtype=float)
        for tr_idx, va_idx in kf.split(p_base, strat):
            shifts = fit_chronic_shifts(y[tr_idx], p_base[tr_idx], chronic[tr_idx], cf)
            pcv[va_idx] = apply_chronic_shifts(p_base[va_idx], chronic[va_idx], shifts)
        mae = float(mean_absolute_error(y, pcv))
        obj = mae + (CFG.PEN_CHRONIC_ON if cf > 0 else 0.0)
        rows.append((obj, mae, cf))
        if best is None or obj < best[0]:
            best = (obj, mae, cf)
    rows.sort(key=lambda x: x[0])
    print("\n[chronic shift tuning] Top 10:")
    for i, (obj, mae, cf) in enumerate(rows[:10], 1):
        print(f"  #{i:02d} obj={obj:.3f} | CV_MAE={mae:.3f} | cf={cf:.2f}")
    obj, mae, cf = best
    return {"base_mae": base_mae, "obj": float(obj), "cv_mae": float(mae), "cf": float(cf), "cv_gain_vs_base": float(base_mae - mae)}

# Baseline shrink (IDENTICAL)
def tune_baseline_shrink(oof_pred, y, baseline):
    grid = np.round(np.arange(0.0, 1.0+1e-9, CFG.SHRINK_GRID_STEP), 10)
    rows = [(float(mean_absolute_error(y, w*oof_pred + (1-w)*baseline)), float(w)) for w in grid]
    rows.sort(key=lambda x: x[0])
    best_mae, best_w = rows[0]
    eligible = [r for r in rows if r[0] <= best_mae + CFG.SHRINK_ONESE_TOL]
    chosen_mae, chosen_w = min(eligible, key=lambda r: (r[1], r[0]))
    return {"best_mae":float(best_mae), "best_w_model":float(best_w),
            "chosen_mae":float(chosen_mae), "chosen_w_model":float(chosen_w),
            "top5": [(float(m), float(w)) for (m,w) in rows[:5]]}

def apply_baseline_shrink(pred, baseline, w_model):
    return w_model*np.asarray(pred, float) + (1-w_model)*np.asarray(baseline, float)

# =============================================================================
# RUN
# =============================================================================
print("="*110)
print("CODE 44 | Code43 + SINGLE CHANGE: Add Model D (Ridge regression) for maximum diversity")
print("="*110)
print(f"[cfg] FAST_MODE={CFG.FAST_MODE} | folds={CFG.N_FOLDS} seeds={CFG.N_SEEDS} | task={CFG.TASK_TYPE}")
print()

must_exist(TRAIN_PATH, "ed_cost_train.csv")
must_exist(TEST_PATH, "ed_cost_test.csv")
must_exist(PATIENTS_PATH, "patients.csv")
must_exist(ADM_TRAIN_PATH, "admissions_train.csv")
must_exist(ADM_TEST_PATH, "admissions_test.csv")
if CFG.REQUIRE_RECEIPTS:
    must_exist(RECEIPTS_JOBLIB_PATH, "receipts_parsed.joblib")

t0 = time.time()
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
patients = pd.read_csv(PATIENTS_PATH)

log_shape("ed_cost_train", train)
log_shape("ed_cost_test", test)

for df in [train, test, patients]:
    df[ID_COL] = pd.to_numeric(df[ID_COL], errors="coerce").astype(int)

adm_df = load_admissions_features(ADM_TRAIN_PATH, ADM_TEST_PATH)
print("  admissions features:", None if adm_df is None else adm_df.shape)

rcpt_df = None
if os.path.exists(RECEIPTS_JOBLIB_PATH):
    rcpt_df = load_receipts_joblib(RECEIPTS_JOBLIB_PATH)
    if rcpt_df is not None:
        rcpt_df["patient_id"] = pd.to_numeric(rcpt_df["patient_id"], errors="coerce").astype("Int64")
        rcpt_df = rcpt_df.dropna(subset=["patient_id"]).copy()
        rcpt_df["patient_id"] = rcpt_df["patient_id"].astype(int)
        rcpt_df = rcpt_df.drop_duplicates("patient_id", keep="last")
        n_feats = rcpt_df.shape[1] - 1
        print(f"  rcpt_df shape: {rcpt_df.shape} | n_features={n_feats}")
        if CFG.REQUIRE_RECEIPTS and n_feats < CFG.MIN_RCPT_FEATURES:
            raise RuntimeError(f"Receipts features too few ({n_feats}). STOP.")
    elif CFG.REQUIRE_RECEIPTS:
        raise RuntimeError("Receipts rcpt_df is None. STOP.")
elif CFG.REQUIRE_RECEIPTS:
    raise FileNotFoundError(f"Missing receipts: {RECEIPTS_JOBLIB_PATH}")

train_feat = build_features(train, patients, adm_df, rcpt_df)
test_feat  = build_features(test,  patients, adm_df, rcpt_df)

# receipts invariant
if rcpt_df is not None and ("rcpt_sum_items" in train_feat.columns):
    diff = train_feat["rcpt_sum_items"].values.astype(float) - train_feat["prior_ed_cost_5y_usd"].values.astype(float)
    match = float(np.mean(np.isfinite(diff) & (np.abs(diff) <= CFG.RCPT_MATCH_TOL)))
    print(f"  Receipt match_rate(train): {match:.4f}")
    if CFG.REQUIRE_RECEIPTS and match < CFG.RCPT_MATCH_RATE_MIN:
        raise RuntimeError(f"Receipt mismatch. STOP.")
if rcpt_df is not None and ("rcpt_sum_items" in test_feat.columns):
    diff = test_feat["rcpt_sum_items"].values.astype(float) - test_feat["prior_ed_cost_5y_usd"].values.astype(float)
    match = float(np.mean(np.isfinite(diff) & (np.abs(diff) <= CFG.RCPT_MATCH_TOL)))
    print(f"  Receipt match_rate(test):  {match:.4f}")

feat_full = drop_constants(get_numeric_feature_cols(train_feat), train_feat)
feat_full = [c for c in feat_full if c not in [TARGET, "rcpt_sum_items"]]

pruned_candidates = [
    "prior_ed_visits_5y","prior_ed_cost_5y_usd","prior_cost_cap20k","sqrt_prior_cost","log_prior_cost","log_prior_cost_cap20k",
    "cost_per_visit","log_visits","baseline_next3y",
    "chronic_encoded","age","sex_encoded","insurance_encoded","zip_region","ins_x_chronic",
    "charlson_max","charlson_mean","pct_emergent",
    "cost_per_em","cost_hhi","pct_cost_em","pct_cost_procedure","pct_cost_critical","pct_cost_high_acuity","pct_cost_imaging","pct_cost_lab",
    "n_high_acuity_total","has_critical_care","has_99285","max_em_level","avg_em_level","n_high_em","n_unique_codes",
    "top1_share","top3_share","gini_line_total","max_line_total",
    "median_unit_price","median_unit_price_em","median_unit_price_imaging","median_unit_price_lab",
    "log1p_median_unit_price","log1p_median_unit_price_em","log1p_median_unit_price_imaging","log1p_median_unit_price_lab","log1p_max_line_total",
    "logprior_x_pctcritical","logprior_x_highacu","cost_per_code",
]
feat_pruned = drop_constants([c for c in pruned_candidates if c in train_feat.columns], train_feat)
feat_pruned = [c for c in feat_pruned if c != "rcpt_sum_items"]

# Ridge features: use a COMPACT subset (most important, avoids multicollinearity)
ridge_features = [
    "prior_ed_cost_5y_usd", "prior_ed_visits_5y", "cost_per_visit",
    "log_prior_cost", "sqrt_prior_cost",
    "chronic_encoded", "age", "sex_encoded", "insurance_encoded", "zip_region",
    "charlson_max", "charlson_mean", "pct_emergent",
    "pct_cost_em", "pct_cost_procedure", "pct_cost_critical", "pct_cost_imaging", "pct_cost_lab",
    "n_high_acuity_total", "has_critical_care", "max_em_level",
    "n_unique_codes", "cost_hhi",
]
ridge_features = [c for c in ridge_features if c in train_feat.columns]

print(f"\n[feature counts] FULL={len(feat_full)} | PRUNED={len(feat_pruned)} | RIDGE={len(ridge_features)}")

# Fill safety
for c in set(feat_full + feat_pruned + ridge_features + ["baseline_next3y"]):
    if c not in train_feat.columns: continue
    med = train_feat[c].median() if not train_feat[c].isna().all() else 0.0
    train_feat[c] = pd.to_numeric(train_feat[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(med)
    if c in test_feat.columns:
        test_feat[c] = pd.to_numeric(test_feat[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(med)

y = train_feat[TARGET].values.astype(float)

tmp = train_feat[["primary_chronic", TARGET]].copy()
tmp["bin"] = pd.qcut(tmp[TARGET], q=5, labels=False, duplicates="drop").astype(str)
tmp["strat"] = tmp["primary_chronic"].astype(str) + "_" + tmp["bin"]
strat_vec = LabelEncoder().fit_transform(tmp["strat"].values)

# Train CatBoost A/B/C
cb_oof, cb_test = train_catboost_models(train_feat, test_feat, feat_full, feat_pruned, strat_vec)

# Train Ridge D
ridge_oof_seeds, ridge_test_seeds = train_ridge_model(train_feat, test_feat, ridge_features, strat_vec)

# Combine into unified dict
oof_by_seed = {
    "A_full_d5": cb_oof["A_full_d5"],
    "B_pruned_d4": cb_oof["B_pruned_d4"],
    "C_pruned_d3": cb_oof["C_pruned_d3"],
    "D_ridge": ridge_oof_seeds,
}
test_by_seed = {
    "A_full_d5": cb_test["A_full_d5"],
    "B_pruned_d4": cb_test["B_pruned_d4"],
    "C_pruned_d3": cb_test["C_pruned_d3"],
    "D_ridge": ridge_test_seeds,
}

# 4-way weight search
wmeta = weight_search_4way(oof_by_seed, y)

# 4-way bag
oof_base, test_base, bag_meta = build_4way_bag_preds(oof_by_seed, test_by_seed, wmeta, y)

base_mae = float(mean_absolute_error(y, oof_base))
print(f"\n[base ensemble after 4-way weight-bagging]")
print(f"  bagged OOF MAE: {base_mae:.3f}")
print("  OOF quantiles:", qdict(oof_base, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)))
print("  TEST quantiles:", qdict(test_base, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)))

# Chronic shift
chronic_train = train_feat["primary_chronic"].astype(str).values
chronic_test  = test_feat["primary_chronic"].astype(str).values

shift_meta = tune_chronic_shift_cv(y, oof_base, chronic_train)
print("\n[best chronic shift]", shift_meta)

apply_shift = (shift_meta["cv_gain_vs_base"] >= CFG.MIN_IMPROVE_TO_APPLY) and (shift_meta["cf"] > 0)
if apply_shift:
    shifts = fit_chronic_shifts(y, oof_base, chronic_train, shift_meta["cf"])
    oof_shift = apply_chronic_shifts(oof_base, chronic_train, shifts)
    test_shift = apply_chronic_shifts(test_base, chronic_test, shifts)
    print("[apply chronic shift] YES |", {k: round(v,3) for k,v in shifts.items()})
else:
    oof_shift = oof_base.copy()
    test_shift = test_base.copy()
    shifts = {}
    print("[apply chronic shift] NO")

mae_shift = float(mean_absolute_error(y, oof_shift))

# Baseline shrink
baseline_tr = train_feat["baseline_next3y"].values.astype(float)
baseline_te = test_feat["baseline_next3y"].values.astype(float)
oof_final = oof_shift.copy()
test_final = test_shift.copy()
shrink_meta = {"applied": False}

if CFG.USE_BASELINE_SHRINK:
    sm = tune_baseline_shrink(oof_shift, y, baseline_tr)
    best_w = sm["chosen_w_model"]
    oof_try = apply_baseline_shrink(oof_shift, baseline_tr, best_w)
    mae_try = float(mean_absolute_error(y, oof_try))
    gain = mae_shift - mae_try
    print(f"\n[baseline shrink] w={best_w:.2f} OOF MAE={mae_try:.3f} (gain {gain:+.3f})")
    if gain >= CFG.MIN_GAIN_TO_APPLY:
        oof_final = oof_try
        test_final = apply_baseline_shrink(test_shift, baseline_te, best_w)
        shrink_meta = {"applied": True, "w_model": float(best_w), "gain": float(gain)}
        print(f"[baseline shrink] APPLY | w={best_w:.2f}")
    else:
        shrink_meta = {"applied": False, "gain": float(gain)}
        print("[baseline shrink] SKIP")

# Clip
y_max = float(np.max(y))
oof_final = np.clip(oof_final, 0.0, y_max * 1.5)
test_final = np.clip(test_final, 0.0, y_max * 1.5)

final_mae = float(mean_absolute_error(y, oof_final))

print("\n" + "="*100)
print("[FINAL SUMMARY - CODE 44]")
print(f"  SINGLE CHANGE: Added Model D (Ridge) to Code43's A/B/C ensemble")
print(f"  base OOF MAE (ABCD bag):   {base_mae:.3f}")
print(f"  after chronic shift MAE:   {mae_shift:.3f}  (delta={mae_shift-base_mae:+.3f})")
print(f"  final OOF MAE:             {final_mae:.3f}")
print(f"  weight meta chosen: wA={wmeta['chosen']['wA']:.2f} wB={wmeta['chosen']['wB']:.2f} wC={wmeta['chosen']['wC']:.2f} wD={wmeta['chosen']['wD']:.2f}")
print(f"  chronic shift: {shift_meta}")
print(f"  baseline shrink: {shrink_meta}")
print("  OOF quantiles:", qdict(oof_final, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)))
print("  TEST quantiles:", qdict(test_final, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)))
print()
print("  >>> Code43: base OOF=427.504, final OOF=427.118, LB=447.9542")
print("  >>> Code31: base OOF=427.928, final OOF=427.480, LB=448.0793")
print("  >>> If wD>0 chosen AND OOF improved -> Ridge adding value.")
print("  >>> If wD=0 chosen -> Ridge not helping, result = Code43.")
print("="*100)

# Write submission
sub = pd.DataFrame({
    "patient_id": test[ID_COL].values.astype(int),
    "ed_cost_next3y_usd": np.round(test_final.astype(float), 2)
})
sub = sub[["patient_id", "ed_cost_next3y_usd"]]
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
sub.to_csv(OUT_SUB_PATH, index=False)

print(f"\n[SUBMISSION] shape={sub.shape} | NaNs={sub['ed_cost_next3y_usd'].isna().any()}")
print(f"  min/med/max: {sub['ed_cost_next3y_usd'].min():.1f} / {sub['ed_cost_next3y_usd'].median():.1f} / {sub['ed_cost_next3y_usd'].max():.1f}")
print(f"\n[done] total wall time: {time.time()-t0:.1f}s")

CODE 44 | Code43 + SINGLE CHANGE: Add Model D (Ridge regression) for maximum diversity
[cfg] FAST_MODE=True | folds=7 seeds=3 | task=CPU

[ed_cost_train] shape=(2000, 5) | cols=5 | mem=0.17 MB
[ed_cost_test] shape=(2000, 4) | cols=4 | mem=0.15 MB
  admissions features: (4000, 4)
  rcpt_df shape: (4000, 45) | n_features=44
  Receipt match_rate(train): 1.0000
  Receipt match_rate(test):  1.0000

[feature counts] FULL=64 | PRUNED=49 | RIDGE=23

[CatBoost training] CPU | folds=7 seeds=3 | iters=3200 es=120
Models: ['A_full_d5', 'B_pruned_d4', 'C_pruned_d3'] | TRIM_K: 0
  Seed 1/3 OOF MAE: A_full_d5=429.99 | B_pruned_d4=430.59 | C_pruned_d3=429.92
  Seed 2/3 OOF MAE: A_full_d5=431.99 | B_pruned_d4=429.53 | C_pruned_d3=429.83
  Seed 3/3 OOF MAE: A_full_d5=432.29 | B_pruned_d4=431.26 | C_pruned_d3=431.11

[CatBoost training] total time: 154.7s
  A_full_d5       seed-agg OOF MAE: 429.20
  B_pruned_d4     seed-agg OOF MAE: 427.98
  C_pruned_d3     seed-agg OOF MAE: 427.74

[Ridge training] feat

# Submissin

In [29]:
# 3. Submit Predictions

# Submit predictions to the competition
print("🚀 Submitting predictions...")

try:
    result = client.submit_prediction("Healthcare", 2, "D:/AgentDs/agent_ds_healthcare/submission.csv")
    
    if result['success']:
        print("✅ Submission successful!")
        print(f"   📊 Score: {result['score']:.4f}")
        print(f"   📏 Metric: {result['metric_name']}")
        print(f"   ✔️  Validation: {'Passed' if result['validation_passed'] else 'Failed'}")
    else:
        print("❌ Submission failed!")
        print(f"   Error details: {result.get('details', {}).get('validation_errors', 'Unknown error')}")
        
except Exception as e:
    print(f"💥 Submission error: {e}")
    print("🔧 Check your API key and team name are correct!")

print("\n🎯 Next steps:")
print("   1. Try incorporating relevant information outside this table!")
print("   2. Move on to Healthcare Challenge 3!")


🚀 Submitting predictions...
✅ Prediction submitted successfully!
📊 Score: 448.2842 (MAE)
✅ Validation passed
✅ Submission successful!
   📊 Score: 448.2842
   📏 Metric: MAE
   ✔️  Validation: Passed

🎯 Next steps:
   1. Try incorporating relevant information outside this table!
   2. Move on to Healthcare Challenge 3!
